# 19 · Top 100 LeetCode questions — part 5: dynamic programming, greedy, intervals and matrices

After this notebook you can solve the 20 most-asked **easy and medium** questions about dynamic programming (DP),
greedy choices, intervals and matrices — and, more importantly, tell *which* of those a brand-new question wants.
DP is the big idea here, so it gets the most pictures: **a table of little answers, each filled in from answers
you already wrote down.** Every answer is plain Python 3, checked with `assert`s, and comes with an animation you
can step through box by box.

**Time:** ~60 min · **Runs:** offline on CPU in seconds · **Needs:** [12 · Dynamic programming](12_dynamic_programming.ipynb), [13 · Greedy and intervals](13_greedy_and_intervals.ipynb)

## How to use this notebook

1. **Read** the story and the question.
2. **Cover the code and try it yourself** — 10 minutes, even if you only get the idea. For DP, try to say the five
   DP lines (below) before you write any code.
3. **Run** the answer, then press **▶ play** on the animation and watch every step. Say out loud *why* each box
   gets its number.
4. **Tomorrow**, solve it again from a blank cell.

Colours in every animation: 🟨 yellow = looking at it now (in DP: *a box we read from*) · 🟩 green = the answer ·
🟥 red = no / blocked · ⬜ grey = finished · 🟦 blue = inside the window · 🟧 orange = moving or swapping ·
🟪 purple = just added (in DP: *the box we just filled*).

## The 20 in this notebook

| # | Problem | LeetCode | Level | Trick |
|---|---|---|---|---|
| 81 | Climbing Stairs | [70](https://leetcode.com/problems/climbing-stairs/) | 🟢 Easy | 1-D DP: ways(i) = ways(i-1) + ways(i-2) |
| 82 | Min Cost Climbing Stairs | [746](https://leetcode.com/problems/min-cost-climbing-stairs/) | 🟢 Easy | 1-D DP: cheapest of the two steps below |
| 83 | House Robber | [198](https://leetcode.com/problems/house-robber/) | 🟡 Medium | 1-D DP: rob it or skip it |
| 84 | House Robber II | [213](https://leetcode.com/problems/house-robber-ii/) | 🟡 Medium | Break the circle: run House Robber twice |
| 85 | Coin Change | [322](https://leetcode.com/problems/coin-change/) | 🟡 Medium | 1-D DP over amounts: 1 + best(amount - coin) |
| 86 | Longest Increasing Subsequence | [300](https://leetcode.com/problems/longest-increasing-subsequence/) | 🟡 Medium | DP "ending here" (+ bisect bonus) |
| 87 | Word Break | [139](https://leetcode.com/problems/word-break/) | 🟡 Medium | DP over prefixes: can I cut here? |
| 88 | Longest Palindromic Substring | [5](https://leetcode.com/problems/longest-palindromic-substring/) | 🟡 Medium | Expand around every centre |
| 89 | Unique Paths | [62](https://leetcode.com/problems/unique-paths/) | 🟡 Medium | 2-D DP: from above + from the left |
| 90 | Longest Common Subsequence | [1143](https://leetcode.com/problems/longest-common-subsequence/) | 🟡 Medium | 2-D DP on two strings |
| 91 | Partition Equal Subset Sum | [416](https://leetcode.com/problems/partition-equal-subset-sum/) | 🟡 Medium | 0/1 knapsack: which sums can I make? |
| 92 | Maximum Subarray | [53](https://leetcode.com/problems/maximum-subarray/) | 🟡 Medium | Kadane: keep the run only if it helps |
| 93 | Jump Game | [55](https://leetcode.com/problems/jump-game/) | 🟡 Medium | Greedy: furthest box I can reach |
| 94 | Jump Game II | [45](https://leetcode.com/problems/jump-game-ii/) | 🟡 Medium | Greedy levels (BFS without a queue) |
| 95 | Gas Station | [134](https://leetcode.com/problems/gas-station/) | 🟡 Medium | Greedy: tank goes negative → start after it |
| 96 | Merge Intervals | [56](https://leetcode.com/problems/merge-intervals/) | 🟡 Medium | Sort by start, glue overlaps |
| 97 | Insert Interval | [57](https://leetcode.com/problems/insert-interval/) | 🟡 Medium | Before · overlapping · after |
| 98 | Non-overlapping Intervals | [435](https://leetcode.com/problems/non-overlapping-intervals/) | 🟡 Medium | Greedy: keep the one that ends first |
| 99 | Rotate Image | [48](https://leetcode.com/problems/rotate-image/) | 🟡 Medium | Transpose, then reverse each row |
| 100 | Spiral Matrix | [54](https://leetcode.com/problems/spiral-matrix/) | 🟡 Medium | Four walls that close in |

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), Path.cwd() / "03_dsa", *Path.cwd().parents]:
    if (_p / "dsa_anim.py").exists():
        sys.path.insert(0, str(_p)); break
from dsa_anim import Movie, arr, bars, hmap, stack, queue, chain, chain_of, tree, grid, graph, note, say
import bisect, itertools, math, random


def timeline(intervals, title="", roles=None, labels=None, end=None):
    """Draw intervals as bars on a clock: row r fills the boxes from its start up to (not including) its end."""
    end = max([e for _, e in intervals] + [1]) if end is None else end
    roles = roles or {}                              # {row: colour}; rows without a colour are drawn with ■
    rows = [["■" if s <= t < e and r not in roles else "" for t in range(end)] for r, (s, e) in enumerate(intervals)]
    hi = {(r, t): roles[r] for r, (s, e) in enumerate(intervals) if r in roles for t in range(s, e)}
    return grid(rows, title, hi=hi, col_labels=list(range(end)),
                row_labels=labels if labels is not None else list(range(len(intervals))))

## The tricks in this notebook, in pictures

**Dynamic programming (DP)** sounds scary but is just this: *a big question is made of smaller copies of itself,
so answer the tiny copies first, write each answer in a box, and build every new box from boxes you already
filled.* Before coding any DP, say these **five DP lines** out loud:

1. **The box** — what does `dp[i]` (or `dp[r][c]`) mean, in one plain sentence?
2. **The rule** — how is a box built from smaller boxes?
3. **The first boxes** — which boxes do you know without any work?
4. **The order** — which way do you fill, so every box you read is already written?
5. **The answer** — which box (or the best of which boxes) is the final answer?

**Greedy** means: at each step, grab the move that looks best *right now* and never look back. It is only right
when you can say *why* the best-looking move can never hurt you later — the third movie shows greedy failing.

**Intervals** are bars on a clock. Almost every interval question starts with **sort** (by start, or by end).

In [ ]:
def stairs_table_movie(n):
    m = Movie("A table of little answers", f"How many ways to climb {n} stairs, taking 1 or 2 steps at a time?")
    heights = list(range(n + 1))
    ways = [None] * (n + 1)
    m.step(f"Big question: how many ways up **{n}** stairs? Too hard at once. So make a row of boxes: one little answer per stair.",
           bars(heights, "the stairs (0 = the floor)"), arr(ways, "ways[i] = ways to stand on stair i"))
    ways[0] = 1
    m.step("Standing on the floor: **1** way (do nothing). We know this box without any work.",
           bars(heights, "the stairs (0 = the floor)", hi={0: "new"}, ptr={"you": 0}),
           arr(ways, "ways[i] = ways to stand on stair i", hi={0: "new"}))
    ways[1] = 1
    m.step("Stair 1: only **1** way — one small step from the floor. Write it down.",
           bars(heights, "the stairs (0 = the floor)", hi={0: "done", 1: "new"}, ptr={"you": 1}),
           arr(ways, "ways[i] = ways to stand on stair i", hi={0: "done", 1: "new"}))
    for i in range(2, n + 1):
        ways[i] = ways[i - 1] + ways[i - 2]          # the rule: your last step came from i-1 or from i-2
        hi = {**{k: "done" for k in range(i - 2)}, i - 2: "look", i - 1: "look", i: "new"}
        m.step(f"Stair {i}: your last step came from stair {i - 1} or stair {i - 2}. Those boxes are **already written**: "
               f"`{ways[i - 1]} + {ways[i - 2]} = {ways[i]}`.",
               bars(heights, "the stairs (0 = the floor)", hi=hi, ptr={"you": i}),
               arr(ways, "ways[i] = ways to stand on stair i", hi=hi))
    m.step(f"The last box holds the big answer: **{ways[n]}** ways. Every box was filled once, from boxes we already had. That is DP!",
           bars(heights, "the stairs (0 = the floor)", hi={n: "good"}),
           arr(ways, "ways[i] = ways to stand on stair i", hi={**{k: "done" for k in range(n)}, n: "good"}))
    return ways[n], m

answer, m = stairs_table_movie(6)
assert answer == 13
m.show()

In [ ]:
def robot_grid_movie(rows, cols, rock):
    m = Movie("A grid of little answers", "A robot moves only right → or down ↓. How many paths reach the bottom-right box, avoiding the rock?")
    ways = [[None] * cols for _ in range(rows)]

    def pic(hi):
        shown = [["🪨" if (r, c) == rock else v for c, v in enumerate(row)] for r, row in enumerate(ways)]
        return grid(shown, "ways[r][c] = paths from the start to this box", hi={**hi, rock: "bad"})

    m.step("Each box will hold a **little answer**: how many paths reach it. Fill row by row, left to right.", pic({}))
    for r in range(rows):
        for c in range(cols):
            older = {(a, b): "done" for a in range(rows) for b in range(cols) if ways[a][b] is not None}
            if (r, c) == rock:
                ways[r][c] = 0                           # nobody can stand on the rock
                continue
            if r == 0 and c == 0:
                ways[r][c] = 1
                m.step("The start box: **1** way — the robot is already there.", pic({(0, 0): "new"}))
                continue
            up = ways[r - 1][c] if r > 0 else 0
            left = ways[r][c - 1] if c > 0 else 0
            ways[r][c] = up + left                       # you arrive from above or from the left
            reads = {k: "look" for k in [(r - 1, c), (r, c - 1)] if min(k) >= 0}
            came = []
            if r > 0:
                came.append("above " + ("(the rock: `0`)" if (r - 1, c) == rock else f"(`{up}`)"))
            if c > 0:
                came.append("the left " + ("(the rock: `0`)" if (r, c - 1) == rock else f"(`{left}`)"))
            m.step(f"You can only arrive from {' or from '.join(came)}. Add them: this box gets **{ways[r][c]}**.",
                   pic({**older, **reads, (r, c): "new"}))
    m.step(f"The bottom-right box says **{ways[-1][-1]}** paths. Each box only needed the box above and the box to its left.",
           pic({(rows - 1, cols - 1): "good"}))
    return ways[-1][-1], m

answer, m = robot_grid_movie(3, 4, rock=(1, 1))
assert answer == 4
m.show()

In [ ]:
def greedy_vs_table_movie(coins, amount):
    m = Movie("Greedy or a table?", f"Pay {amount} with the fewest coins. Coin kinds: {coins}")
    left, hand = amount, []
    m.step(f"**Greedy idea:** always grab the biggest coin that still fits. Left to pay: `{left}`.",
           arr(coins, "coin kinds"), stack(hand, "greedy's hand"), note(left=left))
    for c in sorted(coins, reverse=True):
        while c <= left:
            hand.append(c)
            left -= c
            m.step(f"Biggest coin that fits is `{c}`. Take it. Left to pay: `{left}`.",
                   arr(coins, "coin kinds", hi={coins.index(c): "look"}), stack(hand, "greedy's hand", hi={len(hand) - 1: "new"}), note(left=left))
    greedy = len(hand)
    best = [0] + [math.inf] * amount                 # best[a] = fewest coins that make a
    took = [None] * (amount + 1)
    m.step(f"Greedy used **{greedy}** coins: {' + '.join(map(str, hand))}. Is that really the fewest? Let the table check every choice.",
           stack(hand, "greedy's hand"), arr(best, "best[a] = fewest coins to make a", hi={0: "new"}))
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a and best[a - c] + 1 < best[a]:
                best[a], took[a] = best[a - c] + 1, c
        reads = {a - c: "look" for c in coins if c <= a}
        m.step(f"Amount `{a}`: try every coin, then 1 + the box for what is left. Best is last coin `{took[a]}` → **{best[a]}** coins.",
               arr(best, "best[a] = fewest coins to make a", hi={**{k: "done" for k in range(a)}, **reads, a: "new"}))
    path, a = [], amount
    while a:
        path.append(took[a])
        a -= took[a]
    m.step(f"The table says **{best[amount]}** coins ({' + '.join(map(str, path))}), greedy said {greedy}. "
           "Greedy is only safe when you can say *why* the best-looking move never hurts later.",
           stack(hand, "greedy's hand", hi={k: "bad" for k in range(len(hand))}),
           arr(best, "best[a] = fewest coins to make a", hi={amount: "good"}))
    return best[amount], m

answer, m = greedy_vs_table_movie([1, 3, 4], 6)
assert answer == 2
m.show()

In [ ]:
def busy_blocks_movie(intervals):
    m = Movie("Bars on a clock", "How many separate busy blocks? Sort by start, then sweep from left to right.")
    m.step("Each row is one bar on a clock, filling the boxes from its start up to its end. Jumbled like this, it is hard to read.",
           timeline(intervals, "the bars, as given"), note(bars=len(intervals)))
    bs = sorted(intervals)                           # sort by start time
    m.step("**Sort by start.** Now a bar can only overlap the block we are building, never an older one.",
           timeline(bs, "sorted by start"))
    roles, blocks, end = {}, 0, None
    for i, (s, e) in enumerate(bs):
        if end is not None and s < end:              # starts before the current block ends: same block
            cap = f"Bar {i} starts at `{s}`, before the blue block ends at `{end}`. They **overlap**: it joins the block."
            end = max(end, e)
        else:                                        # a gap: the old block is finished
            roles = {k: "done" for k in roles}
            blocks += 1
            cap = (f"Bar {i} starts at `{s}`" + (f", after the block ended at `{end}`" if end is not None else "") +
                   f". A **new busy block** starts (block {blocks}).")
            end = e
        roles[i] = "win"
        m.step(cap, timeline(bs, "sorted by start", roles={**roles, i: "look"}), note(block_end=end, blocks=blocks))
    m.step(f"Done: **{blocks}** busy blocks. Sorting first meant each bar only had to look at one number: where the current block ends.",
           timeline(bs, "sorted by start", roles={k: "good" for k in roles}), note(blocks=blocks))
    return blocks, m

answer, m = busy_blocks_movie([[8, 10], [1, 4], [12, 14], [3, 6], [9, 11], [5, 7]])
assert answer == 3
m.show()

---

## 81 · Climbing Stairs · [LeetCode 70](https://leetcode.com/problems/climbing-stairs/) · 🟢 Easy

**🧸 Like you're five:** You are climbing a slide's ladder. Each time you can go up 1 rung or 2 rungs. How many different ways
        can you reach the top? The trick: to stand on a rung, your last move came from one rung below or two below.

**The question:** You climb a staircase of `n` steps (`1 ≤ n ≤ 45`), taking 1 or 2 steps at a time. In how many distinct ways can you reach the top?

| Input | Output |
|---|---|
| `n = 2` | `2` |
| `n = 3` | `3` |
| `n = 1` | `1` |

**🔎 How to spot it:** "**how many ways**" to reach the end, where each move is one of a few fixed sizes → 1-D DP, and each box
        is the **sum** of a few boxes behind it. (It is the Fibonacci numbers in disguise.)

**💡 The trick (1-d dp):** The five DP lines:

1. **Box:** `ways[i]` = number of ways to stand on step `i`.
2. **Rule:** `ways[i] = ways[i-1] + ways[i-2]` — your last move was a 1-step or a 2-step.
3. **First boxes:** `ways[0] = 1` (stand on the floor), `ways[1] = 1`.
4. **Order:** left to right, so both boxes you read are already filled.
5. **Answer:** `ways[n]`.

Each box only reads the **two** boxes before it, so keep just two numbers `a, b` instead of the whole row.
The trap: plain recursion without a notebook recomputes the same steps again and again — O(2ⁿ).

**⏱ Cost:** **O(n)** time · **O(1)** space (two numbers). Plain recursion is O(2ⁿ); recursion with a memo (`@cache`) is O(n) time and O(n) space.

In [ ]:
def climb_stairs(n):
    a, b = 1, 1                    # ways to stand on step 0 and on step 1
    for _ in range(n - 1):
        a, b = b, a + b            # slide forward: the new box = the two boxes before it
    return b

assert climb_stairs(1) == 1
assert climb_stairs(2) == 2
assert climb_stairs(3) == 3
assert climb_stairs(5) == 8
assert climb_stairs(45) == 1836311903             # the biggest LeetCode tests
print(climb_stairs(6))

In [ ]:
def climb_stairs_movie(n):
    m = Movie("Climbing Stairs", f"How many ways to climb {n} steps? Keep only the last two boxes.")
    ways = [None] * (n + 1)
    ways[0] = ways[1] = 1
    a, b = 1, 1
    m.step("The first two boxes are free: `1` way to stand on the floor, `1` way to stand on step 1. Call them `a` and `b`.",
           arr(ways, "ways[i]", hi={0: "new", 1: "new"}, ptr={"a": 0, "b": 1}), note(a=a, b=b))
    for i in range(2, n + 1):
        a, b = b, a + b
        ways[i] = b
        m.step(f"Step {i} = the two boxes before it: `{ways[i - 2]} + {ways[i - 1]} = {b}`. Box {i - 2} is never read again, so we can **forget** it.",
               arr(ways, "ways[i]", hi={**{k: "done" for k in range(i - 2)}, i - 2: "done", i - 1: "look", i: "new"},
                   ptr={"a": i - 1, "b": i}), note(a=a, b=b))
    m.step(f"`b` is on the top step: **{b}** ways. We only ever held two numbers, so the space is O(1).",
           arr(ways, "ways[i]", hi={n: "good"}), note(answer=b))
    return b, m

answer, m = climb_stairs_movie(6)
assert answer == climb_stairs(6) == 13
m.show()

---

## 82 · Min Cost Climbing Stairs · [LeetCode 746](https://leetcode.com/problems/min-cost-climbing-stairs/) · 🟢 Easy

**🧸 Like you're five:** Every step on the stairs has a toll: to step off it you pay some candies. You may start on step 0 or
        step 1. What is the fewest candies you must pay to get past the very top?

**The question:** `cost[i]` is the price of leaving step `i`; from there you climb 1 or 2 steps. You may start on step 0 or 1. Return the minimum cost to reach the top (the spot just past the last step). `2 ≤ len(cost) ≤ 1000`.

| Input | Output |
|---|---|
| `cost = [10, 15, 20]` | `15` |
| `cost = [1, 100, 1, 1, 1, 100, 1, 1, 100, 1]` | `6` |
| `cost = [0, 0]` | `0` |

**🔎 How to spot it:** "**cheapest / fewest** way to reach the end" with small fixed moves → 1-D DP with **min** instead of
        sum. Same shape as Climbing Stairs.

**💡 The trick (1-d dp):** The five DP lines:

1. **Box:** `dp[i]` = cheapest way to be *standing on* step `i` (step `n` is the top).
2. **Rule:** `dp[i] = min(dp[i-1] + cost[i-1], dp[i-2] + cost[i-2])` — pay the toll of the step you left.
3. **First boxes:** `dp[0] = dp[1] = 0` (you may start there for free).
4. **Order:** left to right.
5. **Answer:** `dp[n]` — the top is one box *past* the last step.

The trap: answering `dp[n-1]`. The top is not the last step; you still have to leave the last step (or the
one before it).

**⏱ Cost:** **O(n)** time · **O(n)** space for the row (O(1) if you keep two numbers). Trying every path is O(2ⁿ).

In [ ]:
def min_cost_climbing_stairs(cost):
    n = len(cost)
    dp = [0] * (n + 1)             # dp[i] = cheapest way to stand on step i; dp[n] is the top
    for i in range(2, n + 1):
        dp[i] = min(dp[i - 1] + cost[i - 1],     # came up 1 step: paid to leave step i-1
                    dp[i - 2] + cost[i - 2])     # came up 2 steps: paid to leave step i-2
    return dp[n]

assert min_cost_climbing_stairs([10, 15, 20]) == 15
assert min_cost_climbing_stairs([1, 100, 1, 1, 1, 100, 1, 1, 100, 1]) == 6
assert min_cost_climbing_stairs([0, 0]) == 0
assert min_cost_climbing_stairs([5, 3]) == 3

def brute(cost, i=None):                         # try every path from step i to the top
    if i is None:
        return min(brute(cost, 0), brute(cost, 1))
    return 0 if i >= len(cost) else cost[i] + min(brute(cost, i + 1), brute(cost, i + 2))

rng = random.Random(0)
for _ in range(300):
    c = [rng.randint(0, 9) for _ in range(rng.randint(2, 10))]
    assert min_cost_climbing_stairs(c) == brute(c), c
print(min_cost_climbing_stairs([1, 100, 1, 1, 1, 100, 1, 1, 100, 1]))

In [ ]:
def min_cost_movie(cost):
    n = len(cost)
    m = Movie("Min Cost Climbing Stairs", "Pay the toll of each step you leave. Cheapest way past the top?")
    dp = [None] * (n + 1)
    dp[0] = dp[1] = 0
    m.step(f"Box `n = {n}` is the **top**, past the last step. Starting on step 0 or step 1 is free, so both boxes are `0`.",
           arr(cost, "cost[i] = toll to leave step i"), arr(dp, "dp[i] = cheapest to stand on step i", hi={0: "new", 1: "new"}))
    for i in range(2, n + 1):
        one, two = dp[i - 1] + cost[i - 1], dp[i - 2] + cost[i - 2]
        dp[i] = min(one, two)
        pick = i - 1 if one <= two else i - 2
        m.step(f"Reach {'the top' if i == n else f'step {i}'} from step {i - 1} (`{dp[i - 1]} + {cost[i - 1]} = {one}`) "
               f"or from step {i - 2} (`{dp[i - 2]} + {cost[i - 2]} = {two}`). Keep the cheaper: **{dp[i]}**.",
               arr(cost, "cost[i] = toll to leave step i", hi={pick: "look"}),
               arr(dp, "dp[i] = cheapest to stand on step i",
                   hi={**{k: "done" for k in range(i - 2)}, i - 1: "look", i - 2: "look", i: "new"}))
    m.step(f"The top box says **{dp[n]}** candies — the cheap route hops over the `100` tolls.",
           arr(cost, "cost[i] = toll to leave step i", hi={k: "bad" for k in range(n) if cost[k] == max(cost)}),
           arr(dp, "dp[i] = cheapest to stand on step i", hi={n: "good"}))
    return dp[n], m

cost = [1, 100, 1, 1, 100, 1]
answer, m = min_cost_movie(cost)
assert answer == min_cost_climbing_stairs(cost) == 4
m.show()

---

## 83 · House Robber · [LeetCode 198](https://leetcode.com/problems/house-robber/) · 🟡 Medium

**🧸 Like you're five:** Houses stand in a row, each with some candy inside. You may take candy from any houses, but never from
        two houses next to each other, or the alarm rings. How much candy can you get?

**The question:** `nums[i]` is the money in house `i`. You cannot rob two adjacent houses. Return the most money you can rob. `1 ≤ len(nums) ≤ 100`, `0 ≤ nums[i] ≤ 400`.

| Input | Output |
|---|---|
| `nums = [1, 2, 3, 1]` | `4` |
| `nums = [2, 7, 9, 3, 1]` | `12` |
| `nums = [5]` | `5` |

**🔎 How to spot it:** "**pick some items in a row, but not two neighbours**, to get the most" → 1-D DP where each box asks
        **take it or skip it**.

**💡 The trick (1-d dp):** The five DP lines:

1. **Box:** `dp[i]` = the most money from the first `i` houses.
2. **Rule:** `dp[i] = max(dp[i-1], dp[i-2] + nums[i-1])` — skip house `i-1`, or rob it and add the best from
   two houses back.
3. **First boxes:** `dp[0] = 0` (no houses), `dp[1] = nums[0]`.
4. **Order:** left to right.
5. **Answer:** `dp[n]`.

Only the last two boxes are read, so the answer keeps two numbers. The trap: greedily taking every other
house (all the even ones or all the odd ones) — `[2, 1, 1, 9]` wants houses 0 and 3.

**⏱ Cost:** **O(n)** time · **O(1)** space. Trying every allowed set of houses is O(2ⁿ).

In [ ]:
def rob(nums):
    prev2, prev1 = 0, 0            # best from all houses except the last one, best so far
    for x in nums:
        prev2, prev1 = prev1, max(prev1, prev2 + x)   # skip this house, or rob it + the best two back
    return prev1

assert rob([1, 2, 3, 1]) == 4
assert rob([2, 7, 9, 3, 1]) == 12
assert rob([5]) == 5
assert rob([2, 1, 1, 9]) == 11                 # not every-other-house
assert rob([0, 0, 0]) == 0

def brute(nums):                               # every set of houses with no two neighbours
    n, best = len(nums), 0
    for mask in range(1 << n):
        if mask & (mask >> 1) == 0:
            best = max(best, sum(nums[i] for i in range(n) if mask >> i & 1))
    return best

rng = random.Random(0)
for _ in range(300):
    h = [rng.randint(0, 20) for _ in range(rng.randint(1, 10))]
    assert rob(h) == brute(h), h
print(rob([2, 7, 9, 3, 1]))

In [ ]:
def rob_movie(nums):
    n = len(nums)
    m = Movie("House Robber", "Most candy, never two houses side by side")
    dp = [None] * (n + 1)
    dp[0], dp[1] = 0, nums[0]
    m.step(f"Box `dp[i]` = the most candy from the first `i` houses. No houses → `0`. Just house 0 → rob it: `{nums[0]}`.",
           arr(nums, "houses", hi={0: "look"}), arr(dp, "dp[i] = best from the first i houses", hi={0: "new", 1: "new"}))
    for i in range(2, n + 1):
        skip, take = dp[i - 1], dp[i - 2] + nums[i - 1]
        dp[i] = max(skip, take)
        why = "rob it" if take > skip else "skip it"
        m.step(f"House {i - 1} (`{nums[i - 1]}`): rob it → `{nums[i - 1]} + dp[{i - 2}] = {take}`; skip it → `dp[{i - 1}] = {skip}`. "
               f"Better to **{why}**: `{dp[i]}`.",
               arr(nums, "houses", hi={**{k: "done" for k in range(i - 1)}, i - 1: "look"}),
               arr(dp, "dp[i] = best from the first i houses",
                   hi={**{k: "done" for k in range(i - 2)}, i - 2: "look", i - 1: "look", i: "new"}))
    robbed, i = [], n                              # walk back to see which houses we robbed
    while i > 0:
        if i == 1 or dp[i] != dp[i - 1]:
            robbed.append(i - 1)
            i -= 2
        else:
            i -= 1
    m.step(f"Best: **{dp[n]}**, from houses {sorted(robbed)} — no two of them are neighbours.",
           arr(nums, "houses", hi={k: "good" for k in robbed}), arr(dp, "dp[i] = best from the first i houses", hi={n: "good"}))
    return dp[n], m

nums = [2, 7, 9, 3, 1]
answer, m = rob_movie(nums)
assert answer == rob(nums) == 12
m.show()

---

## 84 · House Robber II · [LeetCode 213](https://leetcode.com/problems/house-robber-ii/) · 🟡 Medium

**🧸 Like you're five:** Same candy houses — but now they stand in a **circle**, so the first and the last house are neighbours
        too. Trick: you can never rob both the first and the last. So try the street without the last house, then the
        street without the first house, and keep the better.

**The question:** Houses are in a circle (house 0 and house n-1 are adjacent). No two adjacent houses may be robbed. Return the most money. `1 ≤ len(nums) ≤ 100`.

| Input | Output |
|---|---|
| `nums = [2, 3, 2]` | `3` |
| `nums = [1, 2, 3, 1]` | `4` |
| `nums = [7]` | `7` |

**🔎 How to spot it:** "the same as a known problem, but the ends **wrap around** / it's a **circle**" → break the circle into
        two straight cases and solve each with the old answer.

**💡 The trick (1-d dp, twice):** The five DP lines are House Robber's (`dp[i] = max(dp[i-1], dp[i-2] + house)`), run **twice**:

1. On `nums[:-1]` (the last house is off the table).
2. On `nums[1:]` (the first house is off the table).

Any good plan leaves out the first house or the last house (it cannot rob both), so one of the two runs finds
it. The trap: a single house — `nums[:-1]` and `nums[1:]` are both empty, so return `nums[0]` first.

**⏱ Cost:** **O(n)** time (two passes) · **O(1)** space.

In [ ]:
def rob_ii(nums):
    def rob_line(houses):          # House Robber on a straight street
        prev2 = prev1 = 0
        for x in houses:
            prev2, prev1 = prev1, max(prev1, prev2 + x)
        return prev1

    if len(nums) == 1:             # one house: both slices below would be empty
        return nums[0]
    return max(rob_line(nums[:-1]),    # never touch the last house
               rob_line(nums[1:]))     # never touch the first house

assert rob_ii([2, 3, 2]) == 3
assert rob_ii([1, 2, 3, 1]) == 4
assert rob_ii([1, 2, 3]) == 3
assert rob_ii([7]) == 7
assert rob_ii([2, 7, 9, 3, 1]) == 11            # 2 + 9 in a circle (2 + 9 + 1 would rob two neighbours)

def brute(nums):
    n, best = len(nums), 0
    for mask in range(1 << n):
        ring = mask & (mask >> 1) == 0 and not (n > 1 and mask & 1 and mask >> (n - 1) & 1)
        if ring:
            best = max(best, sum(nums[i] for i in range(n) if mask >> i & 1))
    return best

rng = random.Random(0)
for _ in range(300):
    h = [rng.randint(0, 20) for _ in range(rng.randint(1, 10))]
    assert rob_ii(h) == brute(h), h
print(rob_ii([2, 7, 9, 3, 1]))

In [ ]:
def rob_ii_movie(nums):
    n = len(nums)
    m = Movie("House Robber II", "The houses are in a circle: the first and last are neighbours")
    m.step(f"In a circle, house 0 and house {n - 1} touch, so we can't rob both. Try twice: once **without the last** house, once **without the first**.",
           arr(nums, "houses in a circle", hi={0: "look", n - 1: "look"}))
    results = []
    for name, lo, hi_ in [("without the last house", 0, n - 1), ("without the first house", 1, n)]:
        street = nums[lo:hi_]
        dp = [None] * (len(street) + 1)
        dp[0] = 0
        off = {k: "bad" for k in range(n) if not lo <= k < hi_}
        for i in range(1, len(street) + 1):
            dp[i] = max(dp[i - 1], (dp[i - 2] if i >= 2 else 0) + street[i - 1])
            reads = {i - 1: "look", **({i - 2: "look"} if i >= 2 else {})}
            m.step(f"Run {len(results) + 1} ({name}): house `{street[i - 1]}` → rob it (`{street[i - 1]} + {dp[i - 2] if i >= 2 else 0}`) "
                   f"or skip it (`{dp[i - 1]}`). Best so far **{dp[i]}**.",
                   arr(nums, "houses in a circle", hi={**off, **{lo + k: "done" for k in range(i - 1)}, lo + i - 1: "look"}),
                   arr(dp, f"dp for run {len(results) + 1}", hi={**{k: "done" for k in range(i)}, **reads, i: "new"}))
        results.append(dp[-1])
        m.step(f"Run {len(results)} ({name}) gives **{dp[-1]}**.",
               arr(nums, "houses in a circle", hi=off), arr(dp, f"dp for run {len(results)}", hi={len(street): "good"}))
    best = max(results)
    m.step(f"Keep the better run: `max({results[0]}, {results[1]})` = **{best}**.", arr(nums, "houses in a circle"),
           note(run1=results[0], run2=results[1], answer=best))
    return best, m

nums = [2, 7, 9, 3, 1]
answer, m = rob_ii_movie(nums)
assert answer == rob_ii(nums) == 11
m.show()

---

## 85 · Coin Change · [LeetCode 322](https://leetcode.com/problems/coin-change/) · 🟡 Medium

**🧸 Like you're five:** You must pay 11 cents using the fewest coins from your piggy bank, which only has 2-cent and 5-cent
        coins (as many as you like). Start with tiny amounts: how to pay 1 cent, 2 cents, 3 cents… and use those
        answers to pay the bigger amounts.

**The question:** Given coin values `coins` (unlimited of each) and an `amount`, return the fewest coins that add up to `amount`, or `-1` if it can't be done. `0 ≤ amount ≤ 10⁴`.

| Input | Output |
|---|---|
| `coins = [1, 2, 5], amount = 11` | `3` |
| `coins = [2], amount = 3` | `-1` |
| `coins = [1], amount = 0` | `0` |

**🔎 How to spot it:** "**fewest** items that **add up to** a total, and you may reuse items" → 1-D DP over every amount from
        0 up to the total (the "unbounded knapsack").

**💡 The trick (1-d dp over amounts):** The five DP lines:

1. **Box:** `dp[a]` = fewest coins that make amount `a` (∞ if impossible).
2. **Rule:** `dp[a] = 1 + min(dp[a - c] for every coin c ≤ a)` — whatever coin you pay last, the rest is a
   smaller amount you already solved.
3. **First box:** `dp[0] = 0`.
4. **Order:** amounts from small to big.
5. **Answer:** `dp[amount]`, or `-1` if it is still ∞.

The trap: greedy "biggest coin first" — with coins `[1, 3, 4]` and amount 6 it pays 4+1+1, but 3+3 is
better (see the third pattern movie).

**⏱ Cost:** **O(amount × coins)** time · **O(amount)** space. Trying every mix of coins is exponential.

In [ ]:
def coin_change(coins, amount):
    INF = amount + 1                              # more coins than could ever be needed
    dp = [0] + [INF] * amount                     # dp[a] = fewest coins that make a
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a and dp[a - c] + 1 < dp[a]:  # pay c last; the rest (a - c) is already solved
                dp[a] = dp[a - c] + 1
    return dp[amount] if dp[amount] < INF else -1

assert coin_change([1, 2, 5], 11) == 3
assert coin_change([2], 3) == -1
assert coin_change([1], 0) == 0
assert coin_change([1, 3, 4], 6) == 2                # greedy would say 3
assert coin_change([2, 5], 11) == 4                  # 5 + 2 + 2 + 2

def brute(coins, amount):                        # breadth-first: 1 coin, 2 coins, ...
    level, seen, k = {0}, {0}, 0
    while level:
        if amount in level:
            return k
        level = {s + c for s in level for c in coins if s + c <= amount} - seen
        seen |= level
        k += 1
    return -1

rng = random.Random(0)
for _ in range(300):
    cs = rng.sample(range(1, 12), rng.randint(1, 3))
    a = rng.randint(0, 40)
    assert coin_change(cs, a) == brute(cs, a), (cs, a)
print(coin_change([1, 2, 5], 11))

In [ ]:
def coin_change_movie(coins, amount):
    m = Movie("Coin Change", f"Fewest coins from {coins} to make {amount}")
    dp = [0] + [None] * amount
    m.step(f"Box `dp[a]` = fewest coins to make `a`. Making `0` needs **0** coins. Now fill `1`, `2`, … up to `{amount}`.",
           arr(coins, "coin kinds"), arr(dp, "dp[a] = fewest coins for amount a", hi={0: "new"}))
    for a in range(1, amount + 1):
        dp[a] = math.inf
        tries = []
        for c in coins:
            if c <= a:
                v = dp[a - c] + 1
                tries.append(f"`{c}` + box {a - c} → `{'∞' if v == math.inf else v}`")
                dp[a] = min(dp[a], v)
        reads = {a - c: "look" for c in coins if c <= a}
        if dp[a] == math.inf:
            cap = f"Amount {a}: " + (", ".join(tries) + ". Still impossible" if tries else "no coin is small enough") + ", so **∞**."
        else:
            cap = f"Amount {a}: pay one coin, the rest is a box we know. {', '.join(tries)}. Best: **{dp[a]}**."
        m.step(cap, arr(coins, "coin kinds"),
               arr(dp, "dp[a] = fewest coins for amount a", hi={**{k: "done" for k in range(a)}, **reads, a: "new" if dp[a] < math.inf else "bad"}))
    ans = dp[amount] if dp[amount] < math.inf else -1
    never = [a for a in range(amount + 1) if dp[a] == math.inf]
    m.step(f"The last box: **{ans}** coins make `{amount}`. The ∞ boxes {never} can never be paid with these coins.",
           arr(coins, "coin kinds"), arr(dp, "dp[a] = fewest coins for amount a", hi={amount: "good"}))
    return ans, m

answer, m = coin_change_movie([2, 5], 11)
assert answer == coin_change([2, 5], 11) == 4
m.show()

---

## 86 · Longest Increasing Subsequence · [LeetCode 300](https://leetcode.com/problems/longest-increasing-subsequence/) · 🟡 Medium

**🧸 Like you're five:** Kids stand in a line with number cards. Pick some of them (keep their order, skipping is allowed) so the
        numbers keep going **up**. What is the longest such chain? Each kid asks: "what is the longest going-up chain
        that ends with **me**?"

**The question:** Return the length of the longest strictly increasing subsequence of `nums` (a subsequence keeps the order but may skip elements). `1 ≤ len(nums) ≤ 2500`.

| Input | Output |
|---|---|
| `nums = [10, 9, 2, 5, 3, 7, 101, 18]` | `4` |
| `nums = [0, 1, 0, 3, 2, 3]` | `4` |
| `nums = [7, 7, 7, 7]` | `1` |

**🔎 How to spot it:** "longest / most … **subsequence**" (you may skip items, order kept) → DP where each box means the best
        answer that **ends exactly here**, and you look back at every earlier box.

**💡 The trick (dp "ending here"):** The five DP lines:

1. **Box:** `dp[i]` = length of the longest increasing chain that **ends at** `nums[i]`.
2. **Rule:** `dp[i] = 1 + max(dp[j] for j < i if nums[j] < nums[i])` (just `1` if there is none).
3. **First boxes:** every box starts at `1` — a number alone is a chain.
4. **Order:** left to right; box `i` reads every box to its left.
5. **Answer:** `max(dp)` — the best chain can end anywhere, not only at the last box.

The trap: returning `dp[-1]`. **Bonus, O(n log n):** keep `tails[k]` = the smallest possible last number of a
chain of length `k+1`; for each `x`, `bisect_left(tails, x)` finds the chain it extends (or the tail it
improves). Like patience solitaire: put each card on the leftmost pile whose top is not smaller.

**⏱ Cost:** **O(n²)** time · **O(n)** space for the DP; the bisect version is **O(n log n)** time. Trying every subsequence is O(2ⁿ).

In [ ]:
def length_of_lis(nums):
    dp = [1] * len(nums)                   # dp[i] = longest increasing chain that ENDS at nums[i]
    for i in range(len(nums)):
        for j in range(i):
            if nums[j] < nums[i]:          # nums[i] can go right after nums[j]
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp, default=0)              # the best chain may end anywhere

def length_of_lis_fast(nums):              # bonus: O(n log n) "patience" version
    tails = []                             # tails[k] = smallest last number of any chain of length k+1
    for x in nums:
        k = bisect.bisect_left(tails, x)   # the first pile whose top is >= x
        if k == len(tails):
            tails.append(x)                # x is bigger than every top: a longer chain
        else:
            tails[k] = x                   # same length, but a smaller (better) last number
    return len(tails)

for f in (length_of_lis, length_of_lis_fast):
    assert f([10, 9, 2, 5, 3, 7, 101, 18]) == 4
    assert f([0, 1, 0, 3, 2, 3]) == 4
    assert f([7, 7, 7, 7]) == 1                # strictly increasing: equal numbers don't chain
    assert f([5]) == 1

def brute(nums):                           # try every subsequence
    return max((len(c) for k in range(len(nums) + 1) for c in itertools.combinations(nums, k)
                if all(a < b for a, b in zip(c, c[1:]))), default=0)

rng = random.Random(0)
for _ in range(200):
    xs = [rng.randint(0, 9) for _ in range(rng.randint(1, 9))]
    assert length_of_lis(xs) == length_of_lis_fast(xs) == brute(xs), xs
for _ in range(200):                       # longer lists: the two answers must agree
    xs = [rng.randint(-50, 50) for _ in range(rng.randint(1, 60))]
    assert length_of_lis(xs) == length_of_lis_fast(xs), xs
print(length_of_lis([10, 9, 2, 5, 3, 7, 101, 18]))

In [ ]:
def lis_movie(nums):
    n = len(nums)
    m = Movie("Longest Increasing Subsequence", "The longest chain that keeps going up (skipping is allowed)")
    dp = [None] * n
    m.step("Box `dp[i]` = the longest going-up chain that **ends at** `nums[i]`. A number alone is a chain of `1`.",
           arr(nums, "nums"), arr(dp, "dp[i] = longest chain ending here"))
    for i in range(n):
        dp[i], best_j = 1, None
        for j in range(i):
            if nums[j] < nums[i] and dp[j] + 1 > dp[i]:
                dp[i], best_j = dp[j] + 1, j
        smaller = [j for j in range(i) if nums[j] < nums[i]]
        if best_j is None:
            cap = f"Nothing left of `{nums[i]}` is smaller, so its chain is just itself: **1**."
        else:
            cap = (f"`{nums[i]}` can follow any smaller number to its left (yellow). "
                   f"The best of their boxes is `{dp[best_j]}` (at `{nums[best_j]}`), so `{dp[best_j]} + 1 =` **{dp[i]}**.")
        m.step(cap, arr(nums, "nums", hi={**{j: "look" for j in smaller}, i: "new"}),
               arr(dp, "dp[i] = longest chain ending here", hi={**{j: "done" for j in range(i)}, **{j: "look" for j in smaller}, i: "new"}))
    end = max(range(n), key=lambda k: dp[k])
    chain_ = [end]
    while dp[chain_[-1]] > 1:                   # walk back to rebuild one best chain
        i = chain_[-1]
        chain_.append(next(j for j in range(i) if nums[j] < nums[i] and dp[j] == dp[i] - 1))
    m.step(f"The biggest box is **{dp[end]}** — not the last box! One best chain: {' < '.join(str(nums[k]) for k in reversed(chain_))}.",
           arr(nums, "nums", hi={k: "good" for k in chain_}), arr(dp, "dp[i] = longest chain ending here", hi={end: "good"}))
    return dp[end], m

nums = [10, 9, 2, 5, 3, 7, 101, 18]
answer, m = lis_movie(nums)
assert answer == length_of_lis(nums) == length_of_lis_fast(nums) == 4
m.show()

---

## 87 · Word Break · [LeetCode 139](https://leetcode.com/problems/word-break/) · 🟡 Medium

**🧸 Like you're five:** Someone glued words together with no spaces: `catsanddog`. You have a picture dictionary. Can you cut
        the long string into dictionary words? Walk along it and, at every spot, ask: *could I make a clean cut right
        here?*

**The question:** Return `True` if `s` can be split into a sequence of one or more words from `wordDict` (words may be reused). `1 ≤ len(s) ≤ 300`.

| Input | Output |
|---|---|
| `s = "leetcode", wordDict = ["leet", "code"]` | `True` |
| `s = "applepenapple", wordDict = ["apple", "pen"]` | `True` |
| `s = "catsandog", wordDict = ["cats", "dog", "sand", "and", "cat"]` | `False` |

**🔎 How to spot it:** "can the string be **split / segmented** into pieces from a list" → DP over **prefixes**: `ok[i]` says
        whether the first `i` letters work.

**💡 The trick (dp over prefixes):** The five DP lines:

1. **Box:** `ok[i]` = the first `i` letters can be cut into dictionary words.
2. **Rule:** `ok[i]` is True if some `j < i` has `ok[j]` True **and** `s[j:i]` is a word — a good cut at `j`,
   then one whole word.
3. **First box:** `ok[0] = True` (the empty start is always fine).
4. **Order:** `i` from 1 to `len(s)`.
5. **Answer:** `ok[len(s)]`.

Put the words in a **set** for O(1) lookups, and only look back as far as the longest word. The trap: greedy
"take the first word that fits" — `cat` then `sandog` fails, while `cats` then `and` works.

**⏱ Cost:** **O(n · L)** checks, each slicing up to L letters (L = the longest word) · **O(n)** space. Plain recursion without the table is exponential.

In [ ]:
def word_break(s, word_dict):
    words = set(word_dict)
    longest = max(map(len, words), default=0)
    ok = [True] + [False] * len(s)         # ok[i] = the first i letters can be cut into words
    for i in range(1, len(s) + 1):
        for j in range(max(0, i - longest), i):
            if ok[j] and s[j:i] in words:  # a good cut at j, then one whole word
                ok[i] = True
                break
    return ok[len(s)]

assert word_break("leetcode", ["leet", "code"]) is True
assert word_break("applepenapple", ["apple", "pen"]) is True      # words can be reused
assert word_break("catsandog", ["cats", "dog", "sand", "and", "cat"]) is False
assert word_break("a", ["b"]) is False

def brute(s, words):                       # try every first word, recursively
    return s == "" or any(s.startswith(w) and brute(s[len(w):], words) for w in words)

rng = random.Random(0)
for _ in range(300):
    s = "".join(rng.choice("ab") for _ in range(rng.randint(1, 10)))
    ws = list({"".join(rng.choice("ab") for _ in range(rng.randint(1, 3))) for _ in range(3)})
    assert word_break(s, ws) == brute(s, ws), (s, ws)
print(word_break("catsanddog", ["cat", "cats", "and", "sand", "dog"]))

In [ ]:
def word_break_movie(s, word_dict):
    words, n = set(word_dict), len(s)
    m = Movie("Word Break", f'Can "{s}" be cut into dictionary words?')
    ok = [True] + [None] * n
    marks = lambda: ["✓" if v else ("✗" if v is False else None) for v in ok]
    title = "ok[i]: can the first i letters be cut?"
    m.step("Box `ok[i]` = can the first `i` letters be cut into words? `ok[0]` is **✓**: nothing is cut yet, that's fine.",
           arr(list(s), "letters"), arr(marks(), title, hi={0: "new"}), hmap(words, "dictionary"))
    for i in range(1, n + 1):
        ok[i], found = False, None
        for j in range(i):
            if ok[j] and s[j:i] in words:
                ok[i], found = True, j
                break
        if found is None:
            m.step(f"First {i} letters: no ✓ box `j` is followed by a dictionary word ending here, so `ok[{i}]` = **✗**.",
                   arr(list(s), "letters", hi={k: "done" for k in range(i)}),
                   arr(marks(), title, hi={**{k: "look" for k in range(i) if ok[k]}, i: "bad"}), hmap(words, "dictionary"))
        else:
            w = s[found:i]
            m.step(f"`ok[{found}]` is ✓ and `{w}` is a word, so the first {i} letters can be cut: `ok[{i}]` = **✓**.",
                   arr(list(s), "letters", win=(found, i - 1), hi={k: "done" for k in range(found)}),
                   arr(marks(), title, hi={found: "look", i: "new"}), hmap(words, "dictionary", hi={w: "good"}))
    m.step(f"The last box `ok[{n}]` is **{'✓ True' if ok[n] else '✗ False'}**. (Greedy `cat` + `sand` + `dog` also works here, but `catsandog` would trick it.)",
           arr(list(s), "letters"), arr(marks(), title, hi={n: "good" if ok[n] else "bad"}), hmap(words, "dictionary"))
    return ok[n], m

s, wd = "catsanddog", ["cat", "cats", "and", "sand", "dog"]
answer, m = word_break_movie(s, wd)
assert answer == word_break(s, wd) is True
m.show()

---

## 88 · Longest Palindromic Substring · [LeetCode 5](https://leetcode.com/problems/longest-palindromic-substring/) · 🟡 Medium

**🧸 Like you're five:** A palindrome reads the same both ways, like `racecar`. Put your finger in the middle of a word and
        stretch both arms out one letter at a time; keep going while the two letters match. Do that from every middle
        spot and keep the longest stretch.

**The question:** Return the longest substring of `s` that is a palindrome. If there are several of the same length, any one is accepted. `1 ≤ len(s) ≤ 1000`.

| Input | Output |
|---|---|
| `s = "babad"` | `"bab" (or "aba")` |
| `s = "cbbd"` | `"bb"` |
| `s = "a"` | `"a"` |

**🔎 How to spot it:** "longest **palindromic substring**" (a contiguous piece) → expand around every centre. ("Subsequence"
        would be a 2-D DP instead.)

**💡 The trick (expand around centre):** A palindrome is a mirror around its **centre**. There are `2n - 1` centres: every letter (odd length,
        like `aba`) and every gap between two letters (even length, like `abba`). From each centre, move `L` left and
        `R` right while `s[L] == s[R]`; remember the longest. The trap: forgetting the even centres, so `"cbbd"` gives
        `"c"` instead of `"bb"`. (A DP table `is_pal[i][j] = s[i] == s[j] and is_pal[i+1][j-1]` also works, but needs
        O(n²) space.)

**⏱ Cost:** **O(n²)** time (n centres × up to n steps) · **O(1)** extra space. Checking every substring is O(n³).

In [ ]:
def longest_palindrome(s):
    best_l, best_r = 0, 0                  # ends (inclusive) of the best palindrome so far

    def grow(l, r):                        # stretch both arms while the letters match
        while l >= 0 and r < len(s) and s[l] == s[r]:
            l, r = l - 1, r + 1
        return l + 1, r - 1                # step back to the last matching pair

    for c in range(len(s)):
        for l, r in (grow(c, c), grow(c, c + 1)):   # odd centre (a letter), even centre (a gap)
            if r - l > best_r - best_l:
                best_l, best_r = l, r
    return s[best_l:best_r + 1]

def is_pal(t):
    return t == t[::-1]

assert longest_palindrome("babad") in ("bab", "aba")   # two answers are fine
assert longest_palindrome("cbbd") == "bb"               # an even centre
assert longest_palindrome("a") == "a"
assert len(longest_palindrome("ac")) == 1
assert longest_palindrome("racecar") == "racecar"

rng = random.Random(0)
for _ in range(300):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 12)))
    got = longest_palindrome(s)
    best = max(j - i for i in range(len(s)) for j in range(i + 1, len(s) + 1) if is_pal(s[i:j]))
    assert got in s and is_pal(got) and len(got) == best, (s, got)
print(longest_palindrome("babad"))

In [ ]:
def longest_palindrome_movie(s):
    n = len(s)
    m = Movie("Longest Palindromic Substring", f'Longest mirror-word inside "{s}"')
    best_l, best_r = 0, 0
    m.step("Try every **centre**: each letter, and each gap between two equal letters. From a centre, stretch both arms while the letters match.",
           arr(list(s), "letters"), note(best=s[0]))
    for c in range(n):
        for l, r in ((c, c), (c, c + 1)):
            if r >= n or s[l] != s[r]:
                continue                       # a gap between two different letters grows nothing
            while l - 1 >= 0 and r + 1 < n and s[l - 1] == s[r + 1]:
                l, r = l - 1, r + 1
            found = s[l:r + 1]
            better = r - l > best_r - best_l
            if better:
                best_l, best_r = l, r
            where = f"letter `{s[c]}`" if (r - l) % 2 == 0 else f"the gap between `{s[c]}{s[c + 1]}`"
            stop = "the end of the word" if l == 0 or r == n - 1 else f"`{s[l - 1]}` ≠ `{s[r + 1]}`"
            cap = f"Centre on {where}: arms stretch to `{found}` and stop at {stop}." + (" **New best!**" if better else "")
            centre = {c: "look"} if (r - l) % 2 == 0 else {c: "look", c + 1: "look"}
            m.step(cap, arr(list(s), "letters", win=(l, r), hi=centre, ptr={"L": l, "R": r}), note(best=s[best_l:best_r + 1]))
    ans = s[best_l:best_r + 1]
    m.step(f"Every centre is tried. Longest: **{ans}** (length {len(ans)}) — an even one, found from a gap.",
           arr(list(s), "letters", hi={k: "good" for k in range(best_l, best_r + 1)}), note(best=ans))
    return ans, m

s = "abaxyyxc"
answer, m = longest_palindrome_movie(s)
assert answer == longest_palindrome(s) == "xyyx"
m.show()

---

## 89 · Unique Paths · [LeetCode 62](https://leetcode.com/problems/unique-paths/) · 🟡 Medium

**🧸 Like you're five:** A robot sits in the top-left box of a chocolate bar and wants the bottom-right box. It can only move
        right or down. How many different routes are there? Every box's count is the box above plus the box on the
        left.

**The question:** A robot on an `m × n` grid starts top-left and moves only right or down. How many unique paths reach the bottom-right corner? `1 ≤ m, n ≤ 100` (answer fits in 2·10⁹).

| Input | Output |
|---|---|
| `m = 3, n = 7` | `28` |
| `m = 3, n = 2` | `3` |
| `m = 1, n = 1` | `1` |

**🔎 How to spot it:** "**how many paths** in a grid, moving only right/down" → 2-D DP where each box adds the box above and
        the box to its left.

**💡 The trick (2-d dp):** The five DP lines:

1. **Box:** `ways[r][c]` = number of paths from the start to box `(r, c)`.
2. **Rule:** `ways[r][c] = ways[r-1][c] + ways[r][c-1]` — the last move was down or right.
3. **First boxes:** the whole top row and left column are `1` (only one straight way).
4. **Order:** row by row, left to right — above and left are always ready.
5. **Answer:** `ways[m-1][n-1]`.

Each row only needs the row above, so keep **one row** and update it in place: `row[c] += row[c-1]` (the old
`row[c]` is "above", the new `row[c-1]` is "left"). Maths shortcut: `comb(m+n-2, m-1)`.

**⏱ Cost:** **O(m·n)** time · **O(n)** space (one row). Listing every path is exponential.

In [ ]:
def unique_paths(m, n):
    row = [1] * n                          # the top row: one way to each box (keep going right)
    for _ in range(1, m):
        for c in range(1, n):
            row[c] += row[c - 1]           # from above (the old row[c]) + from the left (the new row[c-1])
    return row[-1]

assert unique_paths(3, 7) == 28
assert unique_paths(3, 2) == 3
assert unique_paths(1, 1) == 1
for m_ in range(1, 13):
    for n_ in range(1, 13):
        assert unique_paths(m_, n_) == math.comb(m_ + n_ - 2, m_ - 1)   # choose which moves are "down"
print(unique_paths(3, 7))

In [ ]:
def unique_paths_movie(rows, cols):
    m = Movie("Unique Paths", f"How many right/down routes cross a {rows} × {cols} grid?")
    ways = [[None] * cols for _ in range(rows)]
    title = "ways[r][c] = routes from the start to here"
    m.step("Each box will hold how many routes reach it. The robot starts top-left 🤖.", grid(ways, title, hi={(0, 0): "look"}))
    for c in range(cols):
        ways[0][c] = 1
    m.step("Top row: the only way is to keep walking right, so every box is **1**.",
           grid(ways, title, hi={(0, c): "new" for c in range(cols)}))
    for r in range(1, rows):
        ways[r][0] = 1
    m.step("Left column: the only way is to keep walking down, so every box is **1** too.",
           grid(ways, title, hi={**{(0, c): "done" for c in range(cols)}, **{(r, 0): "new" for r in range(1, rows)}}))
    for r in range(1, rows):
        for c in range(1, cols):
            ways[r][c] = ways[r - 1][c] + ways[r][c - 1]
            older = {(a, b): "done" for a in range(rows) for b in range(cols) if ways[a][b] is not None}
            m.step(f"Box ({r},{c}): the robot arrived from above (`{ways[r - 1][c]}`) or from the left (`{ways[r][c - 1]}`). "
                   f"`{ways[r - 1][c]} + {ways[r][c - 1]} =` **{ways[r][c]}**.",
                   grid(ways, title, hi={**older, (r - 1, c): "look", (r, c - 1): "look", (r, c): "new"}))
    m.step(f"The bottom-right box: **{ways[-1][-1]}** routes. (The code keeps only one row, because each row only reads the row above.)",
           grid(ways, title, hi={(rows - 1, cols - 1): "good"}))
    return ways[-1][-1], m

answer, m = unique_paths_movie(3, 4)
assert answer == unique_paths(3, 4) == 10
m.show()

---

## 90 · Longest Common Subsequence · [LeetCode 1143](https://leetcode.com/problems/longest-common-subsequence/) · 🟡 Medium

**🧸 Like you're five:** Two kids each have a string of beads. Keep some beads from each (in order, skipping allowed) so both
        strings end up exactly the same. What is the longest string of beads they can both make?

**The question:** Return the length of the longest common subsequence of `text1` and `text2` (0 if none). `1 ≤ lengths ≤ 1000`.

| Input | Output |
|---|---|
| `text1 = "abcde", text2 = "ace"` | `3` |
| `text1 = "abc", text2 = "abc"` | `3` |
| `text1 = "abc", text2 = "def"` | `0` |

**🔎 How to spot it:** **two strings** and "longest common / fewest edits / match them up" → a 2-D table with one string
        down the side and one across the top; `dp[i][j]` answers the question for the first `i` and first `j`
        letters.

**💡 The trick (2-d dp on two strings):** The five DP lines:

1. **Box:** `dp[i][j]` = LCS length of the first `i` letters of `text1` and the first `j` letters of `text2`.
2. **Rule:** if the last letters match, `dp[i][j] = dp[i-1][j-1] + 1` (the box up-left, plus this pair);
   otherwise `dp[i][j] = max(dp[i-1][j], dp[i][j-1])` (drop one letter or the other).
3. **First boxes:** row 0 and column 0 are `0` (an empty string has nothing in common).
4. **Order:** row by row, left to right.
5. **Answer:** `dp[m][n]`, the bottom-right box.

The extra empty row and column save you from `i-1 < 0` checks. The trap: mixing it up with *substring*
(contiguous), where a mismatch resets to 0.

**⏱ Cost:** **O(m·n)** time · **O(m·n)** space (O(min(m, n)) if you keep two rows). Comparing every pair of subsequences is exponential.

In [ ]:
def longest_common_subsequence(text1, text2):
    m, n = len(text1), len(text2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]   # dp[i][j] = LCS of text1[:i] and text2[:j]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if text1[i - 1] == text2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1                 # last letters match: up-left + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])      # drop one letter or the other
    return dp[m][n]

assert longest_common_subsequence("abcde", "ace") == 3
assert longest_common_subsequence("abc", "abc") == 3
assert longest_common_subsequence("abc", "def") == 0
assert longest_common_subsequence("a", "a") == 1

def subseqs(t):
    return {"".join(c) for k in range(len(t) + 1) for c in itertools.combinations(t, k)}

rng = random.Random(0)
for _ in range(200):
    a = "".join(rng.choice("abc") for _ in range(rng.randint(1, 7)))
    b = "".join(rng.choice("abc") for _ in range(rng.randint(1, 7)))
    assert longest_common_subsequence(a, b) == max(map(len, subseqs(a) & subseqs(b))), (a, b)
print(longest_common_subsequence("abcde", "ace"))

In [ ]:
def lcs_movie(text1, text2):
    m_, n_ = len(text1), len(text2)
    m = Movie("Longest Common Subsequence", f'Longest bead string both "{text1}" and "{text2}" can make')
    dp = [[0] * (n_ + 1)] + [[0] + [None] * n_ for _ in range(m_)]
    rl, cl = [None] + list(text1), [None] + list(text2)
    title = "dp[i][j] = LCS of the first i and first j letters"
    pic = lambda hi: grid(dp, title, hi=hi, row_labels=rl, col_labels=cl)
    m.step("Row 0 and column 0 stand for an **empty** string: nothing in common, so they are `0`.",
           pic({**{(0, j): "new" for j in range(n_ + 1)}, **{(i, 0): "new" for i in range(m_ + 1)}}))
    for i in range(1, m_ + 1):
        for j in range(1, n_ + 1):
            a, b = text1[i - 1], text2[j - 1]
            older = {(r, c): "done" for r in range(m_ + 1) for c in range(n_ + 1) if dp[r][c] is not None}
            if a == b:
                dp[i][j] = dp[i - 1][j - 1] + 1
                cap = f"`{a}` = `{b}`: a **match**! Take the box up-left (`{dp[i - 1][j - 1]}`) and add 1: **{dp[i][j]}**."
                reads = {(i - 1, j - 1): "look"}
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
                cap = f"`{a}` ≠ `{b}`: drop one of them — the bigger of above (`{dp[i - 1][j]}`) and left (`{dp[i][j - 1]}`): **{dp[i][j]}**."
                reads = {(i - 1, j): "look", (i, j - 1): "look"}
            m.step(cap, pic({**older, **reads, (i, j): "new"}))
    path, i, j = [], m_, n_                      # walk back to find the shared letters
    while i and j:
        if text1[i - 1] == text2[j - 1]:
            path.append((i, j))
            i, j = i - 1, j - 1
        elif dp[i - 1][j] >= dp[i][j - 1]:
            i -= 1
        else:
            j -= 1
    common = "".join(text1[r - 1] for r, _ in reversed(path))
    m.step(f"The bottom-right box: **{dp[m_][n_]}**. Walking back through the matches (green) spells `{common}`.",
           pic({**{p: "good" for p in path}, (m_, n_): "good"}))
    return dp[m_][n_], m

answer, m = lcs_movie("ace", "abcde")
assert answer == longest_common_subsequence("ace", "abcde") == longest_common_subsequence("abcde", "ace") == 3
m.show()

---

## 91 · Partition Equal Subset Sum · [LeetCode 416](https://leetcode.com/problems/partition-equal-subset-sum/) · 🟡 Medium

**🧸 Like you're five:** You and a friend share a bag of sweets with different weights. Can you split them so you both carry
        exactly the same weight? Each of you must carry half, so the question is: can some sweets weigh exactly half?

**The question:** Return `True` if `nums` (positive integers) can be split into two groups with equal sums. `1 ≤ len(nums) ≤ 200`, `1 ≤ nums[i] ≤ 100`.

| Input | Output |
|---|---|
| `nums = [1, 5, 11, 5]` | `True` |
| `nums = [1, 2, 3, 5]` | `False` |
| `nums = [1, 1]` | `True` |

**🔎 How to spot it:** "**choose some items** (each at most once) to hit an **exact sum**" → 0/1 knapsack: a row of
        True/False boxes, one per possible sum.

**💡 The trick (0/1 knapsack dp):** First: if the total is odd, the answer is `False`. Otherwise `target = total // 2`. The five DP lines:

1. **Box:** `can[s]` = some of the numbers seen so far add up to exactly `s`.
2. **Rule:** when number `x` arrives, `can[s]` becomes True if `can[s - x]` was True (add `x` to that group).
3. **First box:** `can[0] = True` (take nothing).
4. **Order:** numbers one by one; for each number walk the sums **downwards**, from `target` to `x`.
5. **Answer:** `can[target]`.

The trap: walking the sums upwards — then `x` could be used twice in one pass (`can[2]` → `can[4]` with the
same `2`). Walking down reads only boxes from before this number.

**⏱ Cost:** **O(n · target)** time · **O(target)** space. Trying every subset is O(2ⁿ).

In [ ]:
def can_partition(nums):
    total = sum(nums)
    if total % 2:                          # an odd total can't be split in two equal halves
        return False
    target = total // 2
    can = [True] + [False] * target        # can[s] = some numbers so far add up to s
    for x in nums:
        for s in range(target, x - 1, -1): # walk DOWN so each number is used at most once
            if can[s - x]:
                can[s] = True
        if can[target]:                    # found half: stop early
            return True
    return can[target]

assert can_partition([1, 5, 11, 5]) is True
assert can_partition([1, 2, 3, 5]) is False
assert can_partition([1, 1]) is True
assert can_partition([2]) is False                 # one number can't be split
assert can_partition([2, 2, 3, 5]) is False        # total 12, half 6: no subset makes 6

def brute(nums):
    total = sum(nums)
    return total % 2 == 0 and any(sum(c) * 2 == total for k in range(len(nums) + 1)
                                  for c in itertools.combinations(nums, k))

rng = random.Random(0)
for _ in range(300):
    xs = [rng.randint(1, 12) for _ in range(rng.randint(1, 8))]
    assert can_partition(xs) == brute(xs), xs
print(can_partition([1, 5, 11, 5]))

In [ ]:
def can_partition_movie(nums):
    total = sum(nums)
    target = total // 2
    m = Movie("Partition Equal Subset Sum", f"Can some of {nums} add up to exactly half?")
    can = [True] + [False] * target
    made_by = {}                               # sum -> index of the sweet that first made it
    rows, labels = [["✓" if v else "" for v in can]], [None]
    title = "which sums can we make? (one row per number added)"
    pic = lambda hi: grid(rows, title, hi=hi, row_labels=labels, col_labels=list(range(target + 1)))
    m.step(f"The total is `{total}`, so each friend must carry `{target}`. Boxes `0…{target}` say which sums we can make. "
           "With no sweets, only `0` (✓).", arr(nums, "sweets"), pic({(0, 0): "new"}))
    for k, x in enumerate(nums):
        new = [s for s in range(target, x - 1, -1) if can[s - x] and not can[s]]
        for s in range(target, x - 1, -1):
            if can[s - x]:
                if not can[s]:
                    made_by[s] = k
                can[s] = True
        rows.append(["✓" if v else "" for v in can])
        labels.append(x)
        r = len(rows) - 1
        hi = {**{(r - 1, s - x): "look" for s in new}, **{(r, s): "new" for s in new}}
        m.step(f"Add sweet `{x}`: every sum we could make, **plus {x}**, is now makeable: {sorted(new)}. "
               "(Walk the sums downwards so this sweet counts once.)",
               arr(nums, "sweets", hi={**{j: "done" for j in range(k)}, k: "look"}), pic(hi))
        if can[target]:
            m.step(f"Box `{target}` is ✓ — some sweets weigh exactly half! Stop early: **True**.",
                   arr(nums, "sweets", hi={j: "done" for j in range(k + 1)}), pic({(r, target): "good"}))
            mine, s = [], target
            while s:                           # walk back: which sweets made the target?
                mine.append(made_by[s])
                s -= nums[made_by[s]]
            other = [j for j in range(len(nums)) if j not in mine]
            m.step(f"One friend carries {' + '.join(str(nums[j]) for j in sorted(mine))} = `{target}`, "
                   f"the other {' + '.join(str(nums[j]) for j in other)} = `{target}`. Fair! 🍬",
                   arr(nums, "sweets", hi={j: "good" for j in mine}), pic({(r, target): "good"}))
            return True, m
    m.step(f"Box `{target}` never became ✓: **False**.", arr(nums, "sweets"), pic({(len(rows) - 1, target): "bad"}))
    return False, m

nums = [1, 5, 11, 5]
answer, m = can_partition_movie(nums)
assert answer == can_partition(nums) is True
m.show()

---

## 92 · Maximum Subarray · [LeetCode 53](https://leetcode.com/problems/maximum-subarray/) · 🟡 Medium

**🧸 Like you're five:** You walk along a path of coins, some good (+) and some bad (−). You may start and stop wherever you
        like, but must walk a single unbroken stretch. Carry your running total; if it ever drops below zero, it can
        only hurt you, so drop it and start fresh.

**The question:** Return the largest sum of a non-empty contiguous subarray of `nums`. `1 ≤ len(nums) ≤ 10⁵`, values may be negative.

| Input | Output |
|---|---|
| `nums = [-2, 1, -3, 4, -1, 2, 1, -5, 4]` | `6` |
| `nums = [5, 4, -1, 7, 8]` | `23` |
| `nums = [-3, -1, -2]` | `-1` |

**🔎 How to spot it:** "largest (or smallest) **sum of a contiguous subarray**" → Kadane. (Also: "max profit", "best
        streak".)

**💡 The trick (kadane (dp / greedy)):** The five DP lines (Kadane is a tiny DP):

1. **Box:** `cur[i]` = the best sum of a stretch that **ends at** `i`.
2. **Rule:** `cur[i] = max(nums[i], cur[i-1] + nums[i])` — join the old stretch only if it helps.
3. **First box:** `cur[0] = nums[0]`.
4. **Order:** left to right, and only the last box is needed, so keep one number.
5. **Answer:** the biggest `cur` ever seen (`best`).

Why dropping is safe (the greedy sentence): a stretch with a negative total only makes whatever comes next
smaller. The trap: starting `best = 0` — an all-negative list must return its biggest (least negative) number.

**⏱ Cost:** **O(n)** time · **O(1)** space. Trying every stretch is O(n²) (O(n³) if you re-add each one).

In [ ]:
def max_sub_array(nums):
    best = cur = nums[0]                   # not 0: an all-negative list must still pick one number
    for x in nums[1:]:
        cur = max(x, cur + x)              # join the old stretch only if it helps
        best = max(best, cur)
    return best

assert max_sub_array([-2, 1, -3, 4, -1, 2, 1, -5, 4]) == 6
assert max_sub_array([5, 4, -1, 7, 8]) == 23
assert max_sub_array([-3, -1, -2]) == -1
assert max_sub_array([1]) == 1

rng = random.Random(0)
for _ in range(300):
    xs = [rng.randint(-9, 9) for _ in range(rng.randint(1, 12))]
    brute = max(sum(xs[i:j]) for i in range(len(xs)) for j in range(i + 1, len(xs) + 1))
    assert max_sub_array(xs) == brute, xs
print(max_sub_array([-2, 1, -3, 4, -1, 2, 1, -5, 4]))

In [ ]:
def max_sub_array_movie(nums):
    m = Movie("Maximum Subarray", "Biggest total of one unbroken stretch")
    cur = best = nums[0]
    start, best_lr = 0, (0, 0)
    m.step(f"Start the stretch at `{nums[0]}`. Running total `cur = {cur}`, best so far `{best}`.",
           arr(nums, "coins on the path", win=(0, 0), hi={0: "look"}), note(cur=cur, best=best))
    for i in range(1, len(nums)):
        x, old = nums[i], cur
        if cur < 0:                            # same as max(x, cur + x): the old stretch only hurts
            cur, start = x, i
            cap = f"The stretch so far totals `{old}`, below zero — it would only drag `{x}` down. **Drop it**, start fresh at `{x}`."
        else:
            cur = cur + x
            cap = f"Carry the stretch on: `{old} + ({x}) = {cur}`."
        if cur > best:
            best, best_lr = cur, (start, i)
            cap += " **New best!**"
        m.step(cap, arr(nums, "coins on the path", win=(start, i), hi={i: "look"}), note(cur=cur, best=best))
    l, r = best_lr
    m.step(f"Best stretch: `{nums[l:r + 1]}` with total **{best}**.",
           arr(nums, "coins on the path", hi={k: "good" for k in range(l, r + 1)}), note(best=best))
    return best, m

nums = [-2, 1, -3, 4, -1, 2, 1, -5, 4]
answer, m = max_sub_array_movie(nums)
assert answer == max_sub_array(nums) == 6
m.show()

---

## 93 · Jump Game · [LeetCode 55](https://leetcode.com/problems/jump-game/) · 🟡 Medium

**🧸 Like you're five:** Stepping stones in a pond, each with a number: the longest hop you can make from it. Can you reach the
        last stone? Walk along and keep track of the **furthest stone you could ever reach**. If you step onto a stone
        past that line — you're stuck.

**The question:** `nums[i]` is the maximum jump length from index `i`. Starting at index 0, return `True` if you can reach the last index. `1 ≤ len(nums) ≤ 10⁴`.

| Input | Output |
|---|---|
| `nums = [2, 3, 1, 1, 4]` | `True` |
| `nums = [3, 2, 1, 0, 4]` | `False` |
| `nums = [0]` | `True` |

**🔎 How to spot it:** "can you **reach the end**" with jumps of *up to* some length → greedy: track the furthest reachable
        index in one pass (no need to try every jump).

**💡 The trick (greedy reach):** Keep `reach` = the furthest index you can get to so far. For each index `i`: if `i > reach`, you can
        never stand here → `False`. Otherwise `reach = max(reach, i + nums[i])`; once `reach ≥ last index` → `True`.
        **Why greedy is safe:** if you can reach a far stone, you can also reach every stone before it (just hop
        shorter), so one number says everything. The trap: a `0` is only deadly if nothing earlier jumps *over* it.

**⏱ Cost:** **O(n)** time · **O(1)** space. DP "can I reach i?" from every earlier stone is O(n²).

In [ ]:
def can_jump(nums):
    reach = 0                              # the furthest index we can get to so far
    for i, x in enumerate(nums):
        if i > reach:                      # this stone is past anything we can reach: stuck
            return False
        reach = max(reach, i + x)
        if reach >= len(nums) - 1:         # the last stone is within reach
            return True
    return True

assert can_jump([2, 3, 1, 1, 4]) is True
assert can_jump([3, 2, 1, 0, 4]) is False
assert can_jump([0]) is True                   # already on the last stone
assert can_jump([2, 0, 0]) is True             # jump over the zero
assert can_jump([0, 1]) is False

def brute(nums):                               # which stones can we stand on?
    ok = [True] + [False] * (len(nums) - 1)
    for i in range(len(nums)):
        if ok[i]:
            for j in range(i + 1, min(len(nums), i + nums[i] + 1)):
                ok[j] = True
    return ok[-1]

rng = random.Random(0)
for _ in range(300):
    xs = [rng.randint(0, 3) for _ in range(rng.randint(1, 12))]
    assert can_jump(xs) == brute(xs), xs
print(can_jump([2, 3, 1, 1, 4]))

In [ ]:
def can_jump_movie(nums):
    n = len(nums)
    m = Movie("Jump Game", "Can we hop to the last stone?")
    reach = 0
    m.step("Each number is the longest hop from that stone. `reach` = the furthest stone we can get to. At the start: stone `0`.",
           arr(nums, "stones", win=(0, 0), ptr={"reach": 0}), note(reach=reach))
    for i, x in enumerate(nums):
        if i > reach:
            m.step(f"Stone {i} is past our furthest reach (`{reach}`). No hop lands here, so we are **stuck**: `False`.",
                   arr(nums, "stones", win=(0, reach), hi={i: "bad", reach: "bad"}, ptr={"i": i, "reach": reach}), note(reach=reach))
            return False, m
        old = reach
        reach = max(reach, i + x)
        cap = f"From stone {i} we can hop `{x}`, up to stone `{i + x}`. " + (
            f"Further than before: **reach = {reach}**." if reach > old else f"Not past `{old}`, so reach stays.")
        if x == 0:
            cap = f"Stone {i} is a `0` — no hop from here. Reach stays `{reach}`."
        m.step(cap, arr(nums, "stones", win=(0, min(reach, n - 1)), hi={**{k: "done" for k in range(i)}, i: "look"},
                        ptr={"i": i, "reach": min(reach, n - 1)}), note(reach=reach))
        if reach >= n - 1:
            m.step(f"Reach covers the last stone: **True**.", arr(nums, "stones", hi={n - 1: "good"}), note(reach=reach))
            return True, m
    return True, m

nums = [1, 2, 0, 1, 0, 0, 3]
answer, m = can_jump_movie(nums)
assert answer == can_jump(nums) is False
m.show()

---

## 94 · Jump Game II · [LeetCode 45](https://leetcode.com/problems/jump-game-ii/) · 🟡 Medium

**🧸 Like you're five:** Same stepping stones, and this time you can surely reach the end. What is the **fewest** hops? Think in
        rounds: round 1 = every stone one hop can reach, round 2 = every stone two hops can reach, and so on — like
        ripples spreading in water.

**The question:** `nums[i]` is the max jump from `i`; the last index is always reachable. Return the minimum number of jumps from index 0 to the last index. `1 ≤ len(nums) ≤ 10⁴`.

| Input | Output |
|---|---|
| `nums = [2, 3, 1, 1, 4]` | `2` |
| `nums = [2, 3, 0, 1, 4]` | `2` |
| `nums = [0]` | `0` |

**🔎 How to spot it:** "**fewest** jumps / steps" along a line where each spot lets you go *up to* some distance → greedy
        BFS by levels, kept in two numbers (`end` of this round, `far` for the next round).

**💡 The trick (greedy levels):** Walk `i` from 0 to `n-2`. Keep `far` = the furthest any stone in the current round can reach. When `i`
        hits `end` (the last stone of the current round), you **must** jump again: `jumps += 1`, `end = far`.
        **Why greedy is safe:** every stone of a round costs the same number of jumps, so the next round is simply
        "everything up to `far`". The trap: looping to `n-1` — you'd count one extra jump when `end` lands exactly on
        the last stone.

**⏱ Cost:** **O(n)** time · **O(1)** space. DP "fewest jumps to reach i" is O(n²); BFS with a queue is O(n) time but O(n) space.

In [ ]:
def jump(nums):
    jumps = end = far = 0                  # end = last stone of this round, far = reach of the next round
    for i in range(len(nums) - 1):         # never jump FROM the last stone
        far = max(far, i + nums[i])
        if i == end:                       # this round is used up: jump again
            jumps += 1
            end = far
    return jumps

assert jump([2, 3, 1, 1, 4]) == 2
assert jump([2, 3, 0, 1, 4]) == 2
assert jump([0]) == 0
assert jump([1, 2]) == 1
assert jump([2, 3, 1, 1, 2, 4, 1]) == 3

def brute(nums):                           # fewest[i] = fewest jumps to stand on stone i
    fewest = [0] + [math.inf] * (len(nums) - 1)
    for i in range(len(nums)):
        for j in range(i + 1, min(len(nums), i + nums[i] + 1)):
            fewest[j] = min(fewest[j], fewest[i] + 1)
    return fewest[-1]

rng = random.Random(0)
for _ in range(300):
    xs = [rng.randint(1, 4) for _ in range(rng.randint(1, 12))]   # >= 1 so the end is always reachable
    assert jump(xs) == brute(xs), xs
print(jump([2, 3, 1, 1, 4]))

In [ ]:
def jump_movie(nums):
    n = len(nums)
    m = Movie("Jump Game II", "Fewest hops to the last stone")
    jumps = end = far = 0
    lo = 0                                     # first stone of the current round
    m.step("Round 0 is just stone 0 (blue). `far` will track how far the **next** round can reach.",
           arr(nums, "stones", win=(0, 0), ptr={"i": 0}), note(jumps=jumps, end=end, far=far))
    for i in range(n - 1):
        far = max(far, i + nums[i])
        m.step(f"Stone {i} (in round {jumps}) can hop to `{i + nums[i]}`. Furthest for the next round: `far = {far}`.",
               arr(nums, "stones", win=(lo, end), hi={i: "look"}, ptr={"i": i, "far": min(far, n - 1)}),
               note(jumps=jumps, end=end, far=far))
        if i == end:
            jumps += 1
            lo, end = end + 1, far
            m.step(f"Stone {i} is the **last stone of this round**, so we must jump. Round {jumps} = stones `{lo}…{min(end, n - 1)}`. Jumps: **{jumps}**.",
                   arr(nums, "stones", win=(lo, min(end, n - 1)), hi={k: "done" for k in range(lo)}), note(jumps=jumps, end=end, far=far))
    m.step(f"The last stone is inside round {jumps}, so the fewest hops is **{jumps}**.",
           arr(nums, "stones", hi={n - 1: "good"}), note(jumps=jumps))
    return jumps, m

nums = [2, 3, 1, 1, 2, 4, 1]
answer, m = jump_movie(nums)
assert answer == jump(nums) == 3
m.show()

---

## 95 · Gas Station · [LeetCode 134](https://leetcode.com/problems/gas-station/) · 🟡 Medium

**🧸 Like you're five:** Petrol stations stand in a circle. At each one you fill up some petrol, then driving to the next one
        burns some. Where should you start so you never run dry and get all the way round?

**The question:** `gas[i]` is the fuel at station `i`, `cost[i]` the fuel to drive from `i` to `i+1` (circular). Starting with an empty tank, return the start index that completes the circuit, or `-1`. If an answer exists it is unique.

| Input | Output |
|---|---|
| `gas = [1, 2, 3, 4, 5], cost = [3, 4, 5, 1, 2]` | `3` |
| `gas = [2, 3, 4], cost = [3, 4, 3]` | `-1` |
| `gas = [5], cost = [4]` | `0` |

**🔎 How to spot it:** "**circular** route, running total must never go **below zero**, find the start" → one pass with a
        running tank that **restarts** after the spot where it went negative.

**💡 The trick (greedy restart):** 1. If `sum(gas) < sum(cost)` there is not enough fuel in the world → `-1`.
2. Otherwise drive from `start = 0`, adding `gas[i] - cost[i]` to `tank`. If `tank < 0` at station `i`, set
`start = i + 1`, `tank = 0`.
**Why greedy is safe:** if you ran dry between A and B, starting at any station between them is even worse —
you would arrive there with *less* fuel than you had when passing it. So jump the start past B. The trap:
forgetting step 1, which is what makes the final `start` correct.

**⏱ Cost:** **O(n)** time · **O(1)** space. Trying every start and driving the full circle is O(n²).

In [ ]:
def can_complete_circuit(gas, cost):
    if sum(gas) < sum(cost):              # not enough fuel in total: impossible
        return -1
    tank = start = 0
    for i in range(len(gas)):
        tank += gas[i] - cost[i]
        if tank < 0:                      # can't get past i from start, nor from any station in between
            start, tank = i + 1, 0
    return start

assert can_complete_circuit([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]) == 3
assert can_complete_circuit([2, 3, 4], [3, 4, 3]) == -1
assert can_complete_circuit([5], [4]) == 0
assert can_complete_circuit([3, 1, 1], [1, 2, 2]) == 0

def works(gas, cost, s):                  # drive once round the circle from s
    tank = 0
    for k in range(len(gas)):
        i = (s + k) % len(gas)
        tank += gas[i] - cost[i]
        if tank < 0:
            return False
    return True

rng = random.Random(0)
for _ in range(300):
    n = rng.randint(1, 7)
    g = [rng.randint(0, 5) for _ in range(n)]
    c = [rng.randint(0, 5) for _ in range(n)]
    good = [s for s in range(n) if works(g, c, s)]
    got = can_complete_circuit(g, c)
    assert (got == -1) if not good else (got in good), (g, c, got)
print(can_complete_circuit([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]))

In [ ]:
def gas_station_movie(gas, cost):
    n = len(gas)
    gain = [g - c for g, c in zip(gas, cost)]
    m = Movie("Gas Station", "Where do we start so the tank never goes below zero?")
    m.step(f"At each station, fill up `gas`, then burn `cost` to reach the next. Total `gain = gas - cost` is `{sum(gain)}` ≥ 0, so some start works.",
           arr(gas, "gas"), arr(cost, "cost to the next station"), arr(gain, "gain"), note(total=sum(gain)))
    tank = start = 0
    for i in range(n):
        old = tank
        tank += gain[i]
        if tank < 0:
            cap = (f"Station {i}: tank `{old} + ({gain[i]}) = {tank}` — **empty**! Starting anywhere from {start} to {i} fails, "
                   f"so try starting at `{i + 1}`.")
            hi = {k: "bad" for k in range(start, i + 1)}
            m.step(cap, arr(gain, "gain", hi=hi, ptr={"start": start, "i": i}), note(tank=tank, start=start))
            start, tank = i + 1, 0
        else:
            m.step(f"Station {i}: tank `{old} + ({gain[i]}) = {tank}`. Still ≥ 0, keep driving.",
                   arr(gain, "gain", win=(start, i), hi={i: "look"}, ptr={"start": start, "i": i}), note(tank=tank, start=start))
    m.step(f"We made it to the end with tank `{tank}`, and the total is ≥ 0, so the loop closes too. Start at station **{start}**.",
           arr(gas, "gas", hi={start: "good"}), arr(gain, "gain", hi={start: "good"}, ptr={"start": start}), note(start=start))
    return start, m

gas, cost = [1, 2, 3, 4, 5], [3, 4, 5, 1, 2]
answer, m = gas_station_movie(gas, cost)
assert answer == can_complete_circuit(gas, cost) == 3
m.show()

---

## 96 · Merge Intervals · [LeetCode 56](https://leetcode.com/problems/merge-intervals/) · 🟡 Medium

**🧸 Like you're five:** Your calendar has play-dates written as bars on a clock, some overlapping. Squash every group of
        overlapping bars into one long bar, so you can see when you are busy.

**The question:** Given `intervals[i] = [start, end]`, merge all overlapping intervals and return the non-overlapping result. Touching intervals like `[1, 4]` and `[4, 5]` count as overlapping. `1 ≤ len(intervals) ≤ 10⁴`.

| Input | Output |
|---|---|
| `[[1, 3], [2, 6], [8, 10], [15, 18]]` | `[[1, 6], [8, 10], [15, 18]]` |
| `[[1, 4], [4, 5]]` | `[[1, 5]]` |
| `[[1, 4]]` | `[[1, 4]]` |

**🔎 How to spot it:** "**merge / combine overlapping** intervals", "total busy time" → sort by start, then one sweep that
        glues each bar to the last merged bar if it overlaps.

**💡 The trick (sort + sweep):** Sort by start. Walk the bars: if a bar starts **at or before** the end of the last merged bar, stretch
        that bar's end to `max(end, this end)`; otherwise it begins a new merged bar. The traps: using `end = this
        end` instead of `max` (a short bar inside a long one would shrink it — `[1, 10]` then `[2, 3]`), and `<`
        instead of `<=` for touching bars.

**⏱ Cost:** **O(n log n)** time for the sort (the sweep is O(n)) · **O(n)** for the output. Comparing every pair again and again is O(n²) or worse.

In [ ]:
def merge(intervals):
    out = []
    for s, e in sorted(intervals):         # sort by start
        if out and s <= out[-1][1]:        # starts before the last merged bar ends (touching counts)
            out[-1][1] = max(out[-1][1], e)   # stretch it; max, in case this bar is inside it
        else:
            out.append([s, e])             # a gap: a new merged bar
    return out

assert merge([[1, 3], [2, 6], [8, 10], [15, 18]]) == [[1, 6], [8, 10], [15, 18]]
assert merge([[1, 4], [4, 5]]) == [[1, 5]]
assert merge([[1, 4]]) == [[1, 4]]
assert merge([[1, 10], [2, 3]]) == [[1, 10]]          # a bar inside a bar
assert merge([[4, 7], [1, 4]]) == [[1, 7]]            # unsorted input

def brute(intervals):                      # mark every half-step as busy, then read off the bars
    busy = sorted({x / 2 for s, e in intervals for x in range(2 * s, 2 * e + 1)})
    out = []
    for x in busy:
        if out and x - out[-1][1] <= 0.5:
            out[-1][1] = x
        else:
            out.append([x, x])
    return [[int(a), int(b)] for a, b in out]

rng = random.Random(0)
for _ in range(300):
    iv = []
    for _ in range(rng.randint(1, 6)):
        s = rng.randint(0, 15)
        iv.append([s, s + rng.randint(0, 5)])
    assert merge(iv) == brute(iv), iv
print(merge([[1, 3], [2, 6], [8, 10], [15, 18]]))

In [ ]:
def merge_movie(intervals):
    m = Movie("Merge Intervals", "Squash overlapping bars into one")
    E = max(e for _, e in intervals)
    m.step("Each row is a bar on the clock (it fills from its start up to its end). First, **sort by start**.",
           timeline(intervals, "as given", end=E))
    bs = sorted(intervals)
    out = []
    for i, (s, e) in enumerate(bs):
        if out and s <= out[-1][1]:
            old = list(out[-1])
            out[-1][1] = max(out[-1][1], e)
            touch = " (they just touch — that counts)" if s == old[1] else ""
            cap = f"Bar `[{s}, {e}]` starts at `{s}`, not after the last merged bar ends (`{old[1]}`){touch}. **Glue** it on: `{out[-1]}`."
        else:
            out.append([s, e])
            cap = (f"Bar `[{s}, {e}]` starts after the last merged bar ends (`{out[-2][1]}`): a **new** merged bar."
                   if len(out) > 1 else f"The first bar `[{s}, {e}]` starts the first merged bar.")
        m.step(cap, timeline(bs, "sorted by start", roles={**{k: "done" for k in range(i)}, i: "look"}, end=E),
               timeline(out, "merged so far", roles={**{k: "done" for k in range(len(out) - 1)}, len(out) - 1: "new"}, end=E))
    m.step(f"Done: **{out}**.", timeline(bs, "sorted by start", roles={k: "done" for k in range(len(bs))}, end=E),
           timeline(out, "merged", roles={k: "good" for k in range(len(out))}, end=E))
    return out, m

iv = [[6, 8], [1, 3], [2, 5], [8, 9], [11, 13]]
answer, m = merge_movie(iv)
assert answer == merge(iv) == [[1, 5], [6, 9], [11, 13]]
m.show()

---

## 97 · Insert Interval · [LeetCode 57](https://leetcode.com/problems/insert-interval/) · 🟡 Medium

**🧸 Like you're five:** Your calendar is already tidy: sorted bars, none overlapping. A new play-date arrives. Copy the bars
        that finish before it, swallow the bars that bump into it, then copy the rest.

**The question:** `intervals` is sorted by start and non-overlapping. Insert `newInterval`, merging where needed, and return the result (still sorted, non-overlapping). `0 ≤ len(intervals) ≤ 10⁴`.

| Input | Output |
|---|---|
| `[[1, 3], [6, 9]], new = [2, 5]` | `[[1, 5], [6, 9]]` |
| `[[1, 2], [3, 5], [6, 7], [8, 10], [12, 16]], new = [4, 8]` | `[[1, 2], [3, 10], [12, 16]]` |
| `[], new = [5, 7]` | `[[5, 7]]` |

**🔎 How to spot it:** "the list is **already sorted and non-overlapping**, add one" → no need to sort again: three simple
        loops (before · overlapping · after) in O(n).

**💡 The trick (three-phase sweep):** 1. **Before:** copy every bar that ends before the new one starts (`end < new_start`).
2. **Overlap:** while a bar starts at or before the new one ends (`start <= new_end`), swallow it:
`new = [min(starts), max(ends)]`.
3. Add the grown new bar, then **after:** copy the rest.
The trap: forgetting the empty list, or the new bar landing at the very end or very start — the three loops
handle both for free.

**⏱ Cost:** **O(n)** time · **O(n)** for the output. Appending and calling Merge Intervals is O(n log n).

In [ ]:
def insert(intervals, new_interval):
    s, e = new_interval
    out, i, n = [], 0, len(intervals)
    while i < n and intervals[i][1] < s:           # 1. ends before the new bar starts: copy
        out.append(list(intervals[i]))
        i += 1
    while i < n and intervals[i][0] <= e:          # 2. overlaps the new bar: swallow it
        s, e = min(s, intervals[i][0]), max(e, intervals[i][1])
        i += 1
    out.append([s, e])
    out.extend(list(iv) for iv in intervals[i:])   # 3. starts after the new bar ends: copy
    return out

assert insert([[1, 3], [6, 9]], [2, 5]) == [[1, 5], [6, 9]]
assert insert([[1, 2], [3, 5], [6, 7], [8, 10], [12, 16]], [4, 8]) == [[1, 2], [3, 10], [12, 16]]
assert insert([], [5, 7]) == [[5, 7]]
assert insert([[1, 5]], [6, 8]) == [[1, 5], [6, 8]]          # goes at the end
assert insert([[3, 5]], [1, 2]) == [[1, 2], [3, 5]]          # goes at the start
assert insert([[1, 5]], [2, 3]) == [[1, 5]]                  # swallowed whole

def tidy(ivs):                                 # the Merge Intervals answer, as a brute force
    out = []
    for a, b in sorted(ivs):
        if out and a <= out[-1][1]:
            out[-1][1] = max(out[-1][1], b)
        else:
            out.append([a, b])
    return out

rng = random.Random(0)
for _ in range(300):
    base = tidy([[a, a + rng.randint(0, 3)] for a in rng.sample(range(0, 30), rng.randint(0, 6))])
    a = rng.randint(0, 30)
    new = [a, a + rng.randint(0, 6)]
    assert insert(base, new) == tidy(base + [new]), (base, new)
print(insert([[1, 2], [3, 5], [6, 7], [8, 10], [12, 16]], [4, 8]))

In [ ]:
def insert_movie(intervals, new_interval):
    E = max(e for _, e in intervals + [new_interval])
    s, e = new_interval
    out, i, n = [], 0, len(intervals)
    m = Movie("Insert Interval", f"Add the new bar {new_interval} to a tidy calendar")
    labels = list(range(n)) + ["new"]
    pic = lambda roles: timeline(intervals + [[s, e]], "calendar + the new bar", roles={**roles, n: "swap"}, labels=labels, end=E)
    res = lambda: timeline(out, "answer so far", roles={**{k: "done" for k in range(len(out) - 1)}, len(out) - 1: "new"} if out else {}, end=E)
    m.step(f"The calendar is sorted and tidy. The **new** bar `[{s}, {e}]` (orange) has to fit in. Bars it swallows turn orange too.", pic({}), res())
    while i < n and intervals[i][1] < s:
        out.append(list(intervals[i]))
        m.step(f"Bar `{intervals[i]}` ends at `{intervals[i][1]}`, before the new bar starts (`{s}`). **Copy** it.",
               pic({**{k: "done" for k in range(i)}, i: "look"}), res())
        i += 1
    first = i
    while i < n and intervals[i][0] <= e:
        s, e = min(s, intervals[i][0]), max(e, intervals[i][1])
        m.step(f"Bar `{intervals[i]}` starts at `{intervals[i][0]}`, not after the new bar ends. They overlap: the new bar **swallows** it → `[{s}, {e}]`.",
               pic({**{k: "done" for k in range(i)}, i: "swap"}), res())
        i += 1
    out.append([s, e])
    m.step(f"Nothing else bumps into it. Put the grown bar `[{s}, {e}]` in.",
           pic({**{k: "done" for k in range(first)}, **{k: "swap" for k in range(first, i)}}), res())
    for k in range(i, n):
        out.append(list(intervals[k]))
        m.step(f"Bar `{intervals[k]}` starts after the new bar ends (`{e}`). **Copy** it.",
               pic({**{j: "done" for j in range(k)}, k: "look"}), res())
    m.step(f"Done: **{out}**.", pic({k: "done" for k in range(n)}),
           timeline(out, "answer", roles={k: "good" for k in range(len(out))}, end=E))
    return out, m

iv, new = [[1, 2], [3, 5], [6, 7], [8, 10], [12, 16]], [4, 8]
answer, m = insert_movie(iv, new)
assert answer == insert(iv, new) == [[1, 2], [3, 10], [12, 16]]
m.show()

---

## 98 · Non-overlapping Intervals · [LeetCode 435](https://leetcode.com/problems/non-overlapping-intervals/) · 🟡 Medium

**🧸 Like you're five:** Lots of shows at the fair, some at the same time. You want to watch as many as possible without any
        two at once. Always pick the show that **finishes first** — that leaves the most time for the rest. How many
        shows must you skip?

**The question:** Return the minimum number of intervals to remove so the rest do not overlap. Intervals that only touch (`[1, 2]` and `[2, 3]`) do not overlap. `1 ≤ len(intervals) ≤ 10⁵`.

| Input | Output |
|---|---|
| `[[1, 2], [2, 3], [3, 4], [1, 3]]` | `1` |
| `[[1, 2], [1, 2], [1, 2]]` | `2` |
| `[[1, 2], [2, 3]]` | `0` |

**🔎 How to spot it:** "**remove the fewest** / **keep the most** intervals so none overlap" (meetings, shows, balloons) →
        sort by **end**, keep greedily.

**💡 The trick (greedy: earliest end first):** Sort by **end**. Walk the bars keeping `end` = the end of the last bar you kept. If a bar starts at or
        after `end`, keep it (`end = its end`); otherwise remove it (`removed += 1`).
        **Why greedy is safe:** the bar that ends first leaves the most room for everything after it, so keeping it
        can never cost you a later bar. The trap: sorting by **start** — one long early bar would block many short
        ones.

**⏱ Cost:** **O(n log n)** time (the sort) · **O(1)** extra space. Trying every subset is O(2ⁿ).

In [ ]:
def erase_overlap_intervals(intervals):
    removed, end = 0, -math.inf
    for s, e in sorted(intervals, key=lambda iv: iv[1]):   # the one that ENDS first goes first
        if s >= end:                       # fits after the last kept bar (touching is fine): keep it
            end = e
        else:                              # clashes with the last kept bar: remove it
            removed += 1
    return removed

assert erase_overlap_intervals([[1, 2], [2, 3], [3, 4], [1, 3]]) == 1
assert erase_overlap_intervals([[1, 2], [1, 2], [1, 2]]) == 2
assert erase_overlap_intervals([[1, 2], [2, 3]]) == 0
assert erase_overlap_intervals([[1, 100], [2, 3], [3, 4], [4, 5]]) == 1   # sorting by start would keep [1, 100]

def brute(ivs):                            # the biggest subset with no clashes
    for k in range(len(ivs), -1, -1):
        for keep in itertools.combinations(sorted(ivs), k):
            if all(a[1] <= b[0] for a, b in zip(keep, keep[1:])):
                return len(ivs) - k

rng = random.Random(0)
for _ in range(300):
    iv = []
    for _ in range(rng.randint(1, 7)):
        s = rng.randint(0, 10)
        iv.append([s, s + rng.randint(1, 4)])
    assert erase_overlap_intervals(iv) == brute(iv), iv
print(erase_overlap_intervals([[1, 2], [2, 3], [3, 4], [1, 3]]))

In [ ]:
def erase_overlap_movie(intervals):
    m = Movie("Non-overlapping Intervals", "Keep the most shows with no two at once; count the ones we skip")
    E = max(e for _, e in intervals)
    m.step("Each row is a show on the clock. **Sort by end time** — the show that finishes first comes first.",
           timeline(intervals, "as given", end=E))
    bs = sorted(intervals, key=lambda iv: iv[1])
    removed, end, roles = 0, -math.inf, {}
    m.step("Sorted by end. We keep a show if it starts at or after the end of the last show we kept.",
           timeline(bs, "sorted by end", end=E), note(busy_until=end, removed=removed))
    for i, (s, e) in enumerate(bs):
        if s >= end:
            cap = (f"Show `[{s}, {e}]` starts at `{s}`, not before `{end}`: no clash. **Keep** it — busy until `{e}` now."
                   if end != -math.inf else f"Show `[{s}, {e}]` ends first of all. **Keep** it — busy until `{e}`.")
            end, roles[i] = e, "good"
        else:
            removed += 1
            roles[i] = "bad"
            cap = f"Show `[{s}, {e}]` starts at `{s}`, before `{end}`: it clashes with a kept show. **Skip** it (skipped: {removed})."
        m.step(cap, timeline(bs, "sorted by end", roles=roles, end=E), note(busy_until=end, removed=removed))
    m.step(f"We kept {len(bs) - removed} shows and skipped **{removed}**.", timeline(bs, "sorted by end", roles=roles, end=E),
           note(removed=removed))
    return removed, m

iv = [[1, 4], [2, 3], [3, 5], [4, 6], [5, 8], [7, 9]]
answer, m = erase_overlap_movie(iv)
assert answer == erase_overlap_intervals(iv) == 3
m.show()

---

## 99 · Rotate Image · [LeetCode 48](https://leetcode.com/problems/rotate-image/) · 🟡 Medium

**🧸 Like you're five:** You have a square sticker sheet and want to turn it a quarter-turn clockwise without a second sheet.
        Two easy moves do it: flip it over the diagonal (like folding corner to corner), then mirror each row.

**The question:** Rotate the `n × n` matrix 90° clockwise **in place** (do not allocate another matrix). `1 ≤ n ≤ 20`.

| Input | Output |
|---|---|
| `[[1, 2, 3], [4, 5, 6], [7, 8, 9]]` | `[[7, 4, 1], [8, 5, 2], [9, 6, 3]]` |
| `[[5, 1, 9, 11], [2, 4, 8, 10], [13, 3, 6, 7], [15, 14, 12, 16]]` | `[[15, 13, 2, 5], [14, 3, 4, 1], [12, 6, 8, 9], [16, 7, 10, 11]]` |
| `[[1]]` | `[[1]]` |

**🔎 How to spot it:** "rotate / flip a **matrix in place**" → build it from two simple moves: **transpose** (swap `[r][c]`
        with `[c][r]`) and **reverse** rows (or columns).

**💡 The trick (matrix: transpose + reverse):** 1. **Transpose:** for every `r < c`, swap `m[r][c]` and `m[c][r]` — rows become columns.
2. **Reverse each row.**
Clockwise = transpose then reverse rows; counter-clockwise = transpose then reverse *columns* (or reverse rows
first). The trap: looping over **all** `(r, c)` in the transpose — every pair gets swapped twice and nothing
changes. Only swap above the diagonal (`c > r`).

**⏱ Cost:** **O(n²)** time (touch every cell a constant number of times) · **O(1)** extra space. Copying into a new matrix is also O(n²) time but O(n²) space — not allowed here.

In [ ]:
def rotate(matrix):
    n = len(matrix)
    for r in range(n):
        for c in range(r + 1, n):          # only above the diagonal, or every pair swaps back
            matrix[r][c], matrix[c][r] = matrix[c][r], matrix[r][c]
    for row in matrix:
        row.reverse()                      # mirror left-right
    # returns nothing: LeetCode checks the matrix itself

a = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
rotate(a)
assert a == [[7, 4, 1], [8, 5, 2], [9, 6, 3]]
b = [[5, 1, 9, 11], [2, 4, 8, 10], [13, 3, 6, 7], [15, 14, 12, 16]]
rotate(b)
assert b == [[15, 13, 2, 5], [14, 3, 4, 1], [12, 6, 8, 9], [16, 7, 10, 11]]
c = [[1]]
rotate(c)
assert c == [[1]]

rng = random.Random(0)
for _ in range(200):
    n = rng.randint(1, 6)
    x = [[rng.randint(0, 99) for _ in range(n)] for _ in range(n)]
    want = [list(col) for col in zip(*x[::-1])]    # the "new matrix" way, for checking
    rotate(x)
    assert x == want
print(a)

In [ ]:
def rotate_movie(matrix):
    n = len(matrix)
    before = [row[:] for row in matrix]
    m = Movie("Rotate Image", "A quarter-turn clockwise, in place: transpose, then reverse each row")
    diag = {(k, k): "look" for k in range(n)}
    m.step("**Step 1 — transpose:** flip over the diagonal (yellow). Box `(r, c)` swaps with box `(c, r)`.",
           grid(matrix, "the matrix", hi=diag))
    for r in range(n):
        for c in range(r + 1, n):
            matrix[r][c], matrix[c][r] = matrix[c][r], matrix[r][c]
            m.step(f"Swap `{matrix[c][r]}` and `{matrix[r][c]}` across the diagonal.",
                   grid(matrix, "the matrix", hi={**diag, (r, c): "swap", (c, r): "swap"}))
    m.step("Transposed: every row is now a column. **Step 2:** reverse each row.", grid(matrix, "the matrix"))
    for r in range(n):
        matrix[r].reverse()
        m.step(f"Reverse row {r}: `{matrix[r]}`.",
               grid(matrix, "the matrix", hi={**{(k, c): "done" for k in range(r) for c in range(n)}, **{(r, c): "swap" for c in range(n)}}))
    m.step("Done! The first column, read bottom-up, became the top row — a quarter-turn clockwise.",
           grid(before, "before"), grid(matrix, "after", hi={(r, c): "good" for r in range(n) for c in range(n)}))
    return matrix, m

mat = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]]
check = [row[:] for row in mat]
rotate(check)
answer, m = rotate_movie(mat)
assert answer == check == [[13, 9, 5, 1], [14, 10, 6, 2], [15, 11, 7, 3], [16, 12, 8, 4]]
m.show()

---

## 100 · Spiral Matrix · [LeetCode 54](https://leetcode.com/problems/spiral-matrix/) · 🟡 Medium

**🧸 Like you're five:** A snail walks around the edge of a garden bed, eating the plants in a spiral: along the top, down the
        right side, back along the bottom, up the left side — and then round again, one ring further in.

**The question:** Return all elements of an `m × n` matrix in spiral order (clockwise, starting top-left). `1 ≤ m, n ≤ 10`.

| Input | Output |
|---|---|
| `[[1, 2, 3], [4, 5, 6], [7, 8, 9]]` | `[1, 2, 3, 6, 9, 8, 7, 4, 5]` |
| `[[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]` | `[1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7]` |
| `[[1], [2], [3]]` | `[1, 2, 3]` |

**🔎 How to spot it:** "walk a matrix in **spiral / layer** order" (also Spiral Matrix II, rotate by layers) → four walls
        `top, bottom, left, right` that move inwards after each side.

**💡 The trick (matrix: shrinking walls):** Keep four walls. Walk the top row left→right, then `top += 1`; the right column top→bottom, then
        `right -= 1`; the bottom row right→left (**only if** `top <= bottom`), then `bottom -= 1`; the left column
        bottom→top (**only if** `left <= right`), then `left += 1`. Repeat while `top <= bottom and left <= right`.
        The trap: forgetting those two `if` checks — a single leftover row or column gets walked twice.

**⏱ Cost:** **O(m·n)** time (each cell once) · **O(1)** extra space besides the output.

In [ ]:
def spiral_order(matrix):
    out = []
    top, bottom, left, right = 0, len(matrix) - 1, 0, len(matrix[0]) - 1
    while top <= bottom and left <= right:
        for c in range(left, right + 1):           # top row, going right
            out.append(matrix[top][c])
        top += 1
        for r in range(top, bottom + 1):           # right column, going down
            out.append(matrix[r][right])
        right -= 1
        if top <= bottom:                          # a bottom row is still left
            for c in range(right, left - 1, -1):   # bottom row, going left
                out.append(matrix[bottom][c])
            bottom -= 1
        if left <= right:                          # a left column is still left
            for r in range(bottom, top - 1, -1):   # left column, going up
                out.append(matrix[r][left])
            left += 1
    return out

assert spiral_order([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) == [1, 2, 3, 6, 9, 8, 7, 4, 5]
assert spiral_order([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]) == [1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7]
assert spiral_order([[1], [2], [3]]) == [1, 2, 3]     # one column
assert spiral_order([[1, 2, 3]]) == [1, 2, 3]         # one row
assert spiral_order([[7]]) == [7]

def peel(matrix):                          # brute force: take the top row, turn the rest, repeat
    m, out = [row[:] for row in matrix], []
    while m:
        out += m.pop(0)
        m = [list(t) for t in zip(*m)][::-1]   # a quarter-turn counter-clockwise
    return out

rng = random.Random(0)
for _ in range(200):
    r, c = rng.randint(1, 6), rng.randint(1, 6)
    x = [[rng.randint(0, 99) for _ in range(c)] for _ in range(r)]
    assert spiral_order(x) == peel(x), x
print(spiral_order([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]))

In [ ]:
def spiral_movie(matrix):
    m = Movie("Spiral Matrix", "Walk round the edge, then one ring further in")
    out, seen = [], []
    top, bottom, left, right = 0, len(matrix) - 1, 0, len(matrix[0]) - 1

    def take(r, c, cap):
        out.append(matrix[r][c])
        seen.append((r, c))
        m.step(cap, grid(matrix, "the matrix", hi={**{p: "done" for p in seen[:-1]}, (r, c): "look"}),
               arr(out, "answer so far", hi={len(out) - 1: "new"}), note(top=top, bottom=bottom, left=left, right=right))

    m.step("Four walls mark the part we have not eaten yet: `top`, `bottom`, `left`, `right`.",
           grid(matrix, "the matrix"), arr(out, "answer so far"), note(top=top, bottom=bottom, left=left, right=right))
    while top <= bottom and left <= right:
        for c in range(left, right + 1):
            take(top, c, f"Top row, going right → take `{matrix[top][c]}`." if c > left else
                 f"**Top wall** (row {top}): walk right. Take `{matrix[top][c]}`.")
        top += 1
        for r in range(top, bottom + 1):
            take(r, right, f"**Right wall** (column {right}): walk down. Take `{matrix[r][right]}`. The top wall moved down to row {top}."
                 if r == top else f"Going down ↓ take `{matrix[r][right]}`.")
        right -= 1
        if top <= bottom:
            for c in range(right, left - 1, -1):
                take(bottom, c, f"**Bottom wall** (row {bottom}): walk left. Take `{matrix[bottom][c]}`. The right wall moved in to column {right}."
                     if c == right else f"Going left ← take `{matrix[bottom][c]}`.")
            bottom -= 1
        if left <= right:
            for r in range(bottom, top - 1, -1):
                take(r, left, f"**Left wall** (column {left}): walk up. Take `{matrix[r][left]}`. The bottom wall moved up to row {bottom}."
                     if r == bottom else f"Going up ↑ take `{matrix[r][left]}`.")
            left += 1
    m.step(f"The walls crossed, so every box is eaten: **{out}**.",
           grid(matrix, "the matrix", hi={p: "good" for p in seen}), arr(out, "answer", hi={k: "good" for k in range(len(out))}))
    return out, m

mat = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]
answer, m = spiral_movie(mat)
assert answer == spiral_order(mat) == [1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7]
m.show()

## Quiz yourself: which trick?

These are **not** in the top 100, but each one uses a trick from this notebook. Decide the trick before you peek.

1. **Decode Ways** (LeetCode 91, 🟡) — `"226"` can be read as letters `2 2 6`, `22 6` or `2 26` (A=1 … Z=26). How many readings?
   <details><summary>which trick?</summary>1-D DP, exactly like Climbing Stairs: the last piece is one digit or two digits, so <code>ways[i] = ways[i-1] (if the digit isn't 0) + ways[i-2] (if the two digits are 10–26)</code>.</details>
2. **Minimum Path Sum** (LeetCode 64, 🟡) — cheapest walk from top-left to bottom-right of a grid of costs, moving right or down.
   <details><summary>which trick?</summary>2-D DP like Unique Paths, with <code>min</code> instead of <code>+</code>: <code>dp[r][c] = cost[r][c] + min(above, left)</code>.</details>
3. **Coin Change II** (LeetCode 518, 🟡) — how many *different combinations* of coins make the amount?
   <details><summary>which trick?</summary>Unbounded-knapsack DP over amounts, but counting: loop <b>coins outside, amounts inside</b> so each combination is counted once, not once per order.</details>
4. **Edit Distance** (LeetCode 72, 🟡) — fewest inserts, deletes and replaces to turn one word into another.
   <details><summary>which trick?</summary>2-D DP on two strings, like LCS: letters match → take the box up-left; otherwise 1 + the smallest of up, left and up-left.</details>
5. **Maximum Product Subarray** (LeetCode 152, 🟡) — the biggest product of an unbroken stretch.
   <details><summary>which trick?</summary>Kadane, but keep both the biggest <b>and</b> the smallest product ending here — a negative number flips the smallest into the biggest.</details>
6. **Minimum Number of Arrows to Burst Balloons** (LeetCode 452, 🟡) — balloons are intervals; one arrow bursts every balloon it passes through.
   <details><summary>which trick?</summary>Greedy intervals like Non-overlapping Intervals: sort by end, shoot at the end of the first balloon, and only fire a new arrow when a balloon starts after that point.</details>
7. **Assign Cookies** (LeetCode 455, 🟢) — each child needs a cookie at least as big as their greed; feed as many children as possible.
   <details><summary>which trick?</summary>Greedy after sorting both lists: give each child the smallest cookie that is big enough — a bigger cookie is never better spent on a less greedy child.</details>
8. **Spiral Matrix II** (LeetCode 59, 🟡) — fill an `n × n` matrix with 1 … n² in spiral order.
   <details><summary>which trick?</summary>The same four shrinking walls as Spiral Matrix, but write numbers instead of reading them.</details>

## Pattern cheat sheet

| If the question says… | think… | example here |
|---|---|---|
| "how many ways" to reach the end with small moves | 1-D DP, add the boxes behind you | 81 Climbing Stairs |
| "cheapest / fewest" way along a line | 1-D DP with `min` | 82 Min Cost Climbing Stairs, 85 Coin Change |
| "pick items but never two neighbours" | DP: take it or skip it | 83 House Robber |
| "…but the ends wrap around / circle" | solve twice without the first / without the last | 84 House Robber II |
| "longest subsequence" (skip allowed) in one list | DP "the best that ends here", look back | 86 LIS |
| "can the string be split into words" | DP over prefixes, `ok[i]` | 87 Word Break |
| "longest palindromic substring" | expand around 2n−1 centres | 88 |
| "how many paths in a grid" (right/down) | 2-D DP: above + left | 89 Unique Paths |
| two strings, "common / edits / match" | 2-D table, one string on each side | 90 LCS |
| "choose some to hit an exact sum", each once | 0/1 knapsack: True/False per sum, walk down | 91 Partition |
| "largest sum of a contiguous subarray" | Kadane: drop the run when it goes negative | 92 Maximum Subarray |
| "can you reach the end" with jumps of up to k | greedy furthest reach | 93 Jump Game |
| "fewest jumps" | greedy levels (`end`, `far`) | 94 Jump Game II |
| circular route, total must never dip below 0 | restart after the spot where it went negative | 95 Gas Station |
| "merge overlapping intervals" | sort by start, glue to the last one | 96 Merge Intervals |
| "insert into sorted, non-overlapping intervals" | before · overlap · after, no sort | 97 Insert Interval |
| "remove fewest / keep most so none overlap" | sort by **end**, keep greedily | 98 Non-overlapping |
| "rotate / flip a matrix in place" | transpose + reverse | 99 Rotate Image |
| "spiral / layer order" | four walls closing in | 100 Spiral Matrix |

## Say it in an interview

- **Clarify first:** can numbers be negative or zero? Can the list be empty? For intervals — do touching ends
  count as overlapping? For matrices — must it be in place? These questions change the answer (Kadane's
  `best = nums[0]`, `<` vs `<=`).
- **Brute force out loud, then spot the repeats:** "Trying every choice is O(2ⁿ), but the same smaller question
  comes up again and again — so I'll store each answer once." That sentence *is* the move to DP.
- **Say the five DP lines before coding:** what the box means, the rule, the first boxes, the fill order, where
  the answer is. Then shrink the space ("each row only reads the row above, so one row is enough").
- **For greedy, give the one-sentence reason it's safe** ("the show that ends first leaves the most room"). If you
  can't, it probably isn't safe — test it on a tiny counter-example like coins `[1, 3, 4]` for 6.
- **Complexity with the reason:** "O(amount × coins) time, O(amount) space, because each box tries every coin once."
- **Common traps:** `dp[-1]` instead of `max(dp)` (LIS), forgetting even centres (palindromes), walking the sums
  upwards in 0/1 knapsack, `end = e` instead of `max(...)` when merging, sorting by start when you should sort by
  end, and walking a single leftover row twice in a spiral.

## Next

That's the whole top 100. 🎉

- Previous: [18 · Top 100, part 4 — heaps, backtracking and graphs](18_top100_part4_heaps_backtracking_graphs.ipynb)
- Go round again, from blank cells: [15 · Top 100, part 1 — arrays, hashing and two pointers](15_top100_part1_arrays_hashing_two_pointers.ipynb)
  — this time cover every answer and only peek at the trick.
- In depth: [12 · Dynamic programming](12_dynamic_programming.ipynb) · [13 · Greedy and intervals](13_greedy_and_intervals.ipynb)